# French exit PS3 — Offline Geocoding from Raw Field Data

**Submission notebook · original datasets → final geocoding outputs**

**Run instructions:** Extract the original `dataset.rar` once. Place this notebook next to the extracted `dataset/` directory (containing `addresses.csv`, `field_visits.csv`, etc.), then select **Restart Kernel → Run All**. The code runs entirely offline; fitting the original model and nested confidence calibration may take several minutes. Requires Python 3.10+, pandas, NumPy, scikit-learn and matplotlib. `rapidfuzz` is optional.

**Outputs:** `submission_outputs/final_predictions.csv`, quality diagnostics, `visit_planner_features.csv`, `ps2_location_features.csv`, and manager charts. Supporting intermediate artifacts are generated automatically *during* the run from the original data.

In [ ]:
from pathlib import Path
import sys, subprocess, shutil, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
required = ['addresses.csv','accounts.csv','field_visits.csv','visit_gps_points.csv',
            'landmarks_poi.csv','baseline_geocodes.csv','localities.csv','surveyed_addresses.csv',
            'splits.csv','towns.csv','agents.csv','lenders.csv']

candidates = [ROOT/'dataset', ROOT/'dataset'/'dataset', ROOT/'raw_ps_data'/'dataset', ROOT]
candidates += [p.parent for p in ROOT.glob('**/addresses.csv') if len(p.parts) <= len(ROOT.parts)+4]
raw_dir = next((d for d in candidates if all((d/f).is_file() for f in required)), None)
if raw_dir is None:
    raise FileNotFoundError('Extract the ORIGINAL dataset.rar next to this notebook as dataset/. '+
                            'The folder must contain addresses.csv, field_visits.csv, and the other raw files.')
print('Raw dataset:', raw_dir.resolve())
print('Input datasets:', ', '.join(required))
print('Addresses:', len(pd.read_csv(raw_dir/'addresses.csv')),
      '| Visits:', len(pd.read_csv(raw_dir/'field_visits.csv')))

def run_script(script, *args):
    cmd = [sys.executable, str(ROOT/script), *[str(arg) for arg in args]]
    print('Running', script, flush=True)
    subprocess.run(cmd, cwd=ROOT, check=True)

shutil.copyfile(raw_dir/'surveyed_addresses.csv', ROOT/'surveyed_addresses.csv')

## 1. Address parsing and GPS-trail integrity

The provided preprocessing algorithm normalizes multilingual text, identifies streets, localities and landmark families, analyzes agent GPS trails, flags suspicious visit evidence, and constructs robust clean-visit pseudo-truth. 

In [ ]:
%%writefile cn_stage1_preprocess.py

# CN PS3 - Stage 1: labeling sample, address parsing (P1-P4), integrity signals.

from __future__ import annotations

import argparse
import json
import re
import sys
import unicodedata
from collections import Counter
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from rapidfuzz import fuzz as _rf

    def sim(a, b):
        return _rf.ratio(a, b)
    FUZZ_BACKEND = "rapidfuzz"
except Exception:  
    from difflib import SequenceMatcher

    def sim(a, b):
        return 100.0 * SequenceMatcher(None, a, b).ratio()
    FUZZ_BACKEND = "difflib (slower; pip install rapidfuzz)"

DATA_DIR = "."
OUT_DIR = "ps3_out"
SEED = 42
N_LABEL = 100
N_LABELERS = 3
LABELS_FILLED = "label_sample_filled.csv"
EXTRA_DICT = "dict_extra.json"
EVAL_HALF = "dev"

FUZZY_ACCEPT = 88.0          
FUZZY_MARGIN = 5.0           
DBSCAN_EPS_M = 75
DBSCAN_MIN_SAMPLES = 10
HUB_MIN_VISITS = 50
HUB_MIN_SPREAD_M = 1500.0    
PHOTO_ADDR_THRESHOLD = 3     
RISK_W = {"hub": 0.40, "photo": 0.30, "tiny": 0.15, "start": 0.15}

def hdr(title):
    print("\n" + "=" * 10 + f" {title} " + "=" * 10)

def find_file(name):
    root = Path(DATA_DIR)
    hits = sorted(root.rglob(name))
    if not hits:
        raise FileNotFoundError(f"{name} not found under {root.resolve()}")
    return hits[0]

def validate_all_datasets_present():
    expected = [
        "accounts.csv", "addresses.csv", "agents.csv", "baseline_geocodes.csv",
        "field_visits.csv", "landmarks_poi.csv", "lenders.csv", "localities.csv",
        "splits.csv", "surveyed_addresses.csv", "towns.csv", "visit_gps_points.csv"
    ]
    for name in expected:
        find_file(name)

def load(name, **kw):
    p = find_file(name)
    return pd.read_csv(p, **kw)

def pick_col(df, candidates, what, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"Could not find a {what} column. Columns present: {list(df.columns)}")
    return None

def detect_text_col(df):
    for c in ["address_text", "raw_address", "address_raw", "address", "full_address", "address_line", "text"]:
        if c in df.columns:
            return c
    obj = [c for c in df.columns if df[c].dtype == object and not c.endswith("_id")]
    best = max(obj, key=lambda c: df[c].astype(str).str.len().mean())
    print(f"[warn] guessed the address text column: '{best}' (columns: {list(df.columns)})")
    return best

def dist(x1, y1, x2, y2):
    return np.hypot(np.asarray(x1, float) - np.asarray(x2, float), np.asarray(y1, float) - np.asarray(y2, float))

def err_stats(a):
    a = np.asarray(a, float)
    a = a[~np.isnan(a)]
    if len(a) == 0:
        return dict(n=0, mean=np.nan, median=np.nan, p90=np.nan, max=np.nan)
    return dict(n=len(a), mean=a.mean(), median=np.median(a), p90=np.percentile(a, 90), max=a.max())

def PH(phrase):
    return r"(?<![a-z0-9])" + re.escape(phrase) + r"(?![a-z0-9])"

ABBR = {
    "nr": "near", "opp": "opposite", "oppo": "opposite", "oppt": "opposite",
    "bhd": "behind", "beh": "behind",
    "govt": "government", "gov": "government", "gvt": "government", "sch": "school",
    "ngr": "nagar", "nagr": "nagar", "ngar": "nagar", "nager": "nagar",
    "colny": "colony", "coloney": "colony", "colni": "colony", "col": "colony",
    "lyt": "layout", "layot": "layout", "lout": "layout",
    "ext": "extension", "extn": "extension",
    "rd": "road", "hno": "house no", "dno": "door no",
    "mohla": "mohalla", "muhalla": "mohalla",
}

VOCAB = ["cross", "main", "road", "street", "gali", "ward", "block", "near", "opposite", "behind", "beside",
         "temple", "school", "masjid", "mosque", "church", "colony", "nagar", "layout", "extension",
         "dairy", "store", "shop", "office", "stop", "tank", "hall", "park", "pump", "bunk", "petrol",
         "medical", "ration", "community", "government", "village", "house"]
VOCAB_SET = set(VOCAB)

GENERIC = {"nagar", "colony", "layout", "extension", "puri", "mohalla", "vihar", "enclave", "park", "road",
           "street", "town", "city", "garden", "gardens", "block", "sector", "stage", "phase", "village",
           "camp", "basti", "east", "west", "north", "south"}

REL_PRE_TOKENS = {"near", "opposite", "behind", "beside", "adjacent"}
TYPE_HEAD_TOKENS = {"temple", "mandir", "gudi", "school", "masjid", "mosque", "church", "park", "tank",
                    "dairy", "shop", "store", "hall", "office", "stop", "stand", "pump", "bunk", "garden"}

REL_WORDS = ["near", "opposite", "behind", "beside", "next to", "close to", "adjacent to", "opp to",
             "hattira", "hatthira", "hathira", "hattirada", "ke paas", "paas", "samne", "ke samne",
             "ke peeche", "peeche", "ke pichhe", "pichhe", "pakka", "pakkada", "pakkadalli", "edurige",
             "eduru", "edurugade", "mundhe", "hinde", "ಹತ್ತಿರ", "ಎದುರು", "ಪಕ್ಕ", "ಹಿಂದೆ"]
REL_RE = re.compile(r"(?<![a-z0-9])(?:%s)(?![a-z0-9])" % "|".join(
    re.escape(w) for w in sorted(REL_WORDS, key=len, reverse=True)))

LM_TERMS = {
    "ration_shop": [r"ration (?:shop|store|depot|dukan|dukaan|angadi)", r"(?:rashan|rasan) (?:dukan|dukaan|shop|store)",
                    r"fair price (?:shop|store)", r"padithara angadi", "ಪಡಿತರ"],
    "bus_stop": [r"bus (?:stop|stand|stn|station|nildana|nilduna|nildhana|shelter)", r"busstop", "ಬಸ್ ನಿಲ್ದಾಣ"],
    "govt_school": [r"(?:government|sarkari|sarakari) (?:high |primary |higher primary |urdu )?(?:school|shaale|shala|shale|pathshala|paathshala|vidyalaya)",
                    "ಸರ್ಕಾರಿ ಶಾಲೆ"],
    "masjid": [r"masjid", r"mosque", r"masidi", r"masjeed", r"masjith", "ಮಸೀದಿ"],
    "hanuman_temple": [r"(?:hanuman|haanuman|hanumaan|hanumantha|anjaneya|maruti)(?: swamy| swami| ji ka| ji| ka)? (?:temple|mandir|gudi|devasthana)",
                       "ಹನುಮಾನ್ ದೇವಸ್ಥಾನ", "ಆಂಜನೇಯ ದೇವಸ್ಥಾನ", "ಆಂಜನೇಯ ಗುಡಿ"],
    "ganesha_temple": [r"(?:ganesh|ganesha|ganapati|ganapathi|vinayaka|vinayak|pillayar)(?: ji ka| ji| ka)? (?:temple|mandir|gudi|devasthana)",
                       "ಗಣೇಶ ದೇವಸ್ಥಾನ", "ಗಣೇಶ ಗುಡಿ"],
    "medical_store": [r"medical (?:store|stores|shop|hall)", r"medicals", r"pharmacy", r"chemist",
                      r"(?:dawai|dawa|davai) (?:ki |ka )?(?:dukan|dukaan|shop)", r"medicine (?:shop|store)",
                      r"(?:aushadhi|oushadhi|aushadh) angadi", "ಮೆಡಿಕಲ್"],
    "water_tank": [r"water (?:tank|tower)", r"overhead tank", r"oht", r"(?:paani|pani) (?:ki )?tanki",
                   r"neeru tank", r"neerina tank", "ನೀರಿನ ಟ್ಯಾಂಕ್"],
    "milk_dairy": [r"milk (?:dairy|dair|booth|parlour|parlor|centre|center)", r"(?:dudh|doodh|halina|haalina) (?:dairy|dair)",
                   r"dairy", r"nandini (?:milk )?(?:parlour|parlor|booth)", "ಹಾಲಿನ ಡೈರಿ", "ಡೈರಿ"],
    "community_hall": [r"community (?:hall|centre|center|bhavan|bhavana)", r"samudaya bhavana?", r"samaj bhavan",
                       r"comm hall", "ಸಮುದಾಯ ಭವನ"],
    "park": [r"park", r"udyanavana", r"udyana", r"garden", r"bagicha", r"bageecha", "ಉದ್ಯಾನ", "ಪಾರ್ಕ್"],
    "church": [r"church", r"chapel", r"girja ?ghar", "ಚರ್ಚ್"],
    "post_office": [r"post office", r"dak ?ghar", r"anche kacheri", r"ancheri", "ಅಂಚೆ ಕಚೇರಿ"],
    "petrol_bunk": [r"petrol (?:bunk|pump|bank|punk)", r"fuel station", r"gas station", "ಪೆಟ್ರೋಲ್ ಬಂಕ್"],
}
LM_RE = {}

R = {}  
PROTECT = set()  

ORD_WORDS = {"first": 1, "second": 2, "third": 3, "fourth": 4, "fifth": 5, "sixth": 6,
             "seventh": 7, "eighth": 8, "ninth": 9, "tenth": 10}
KANNADA_RE = re.compile(r"[\u0C80-\u0CFF]")
DEVANAGARI_RE = re.compile(r"[\u0900-\u097F]")

@lru_cache(maxsize=None)
def repair_token(t):
    if (not t.isalpha()) or (not t.isascii()) or len(t) < 5 or t in PROTECT or t in VOCAB_SET:
        return t
    best, bs = None, 0.0
    for w in VOCAB:
        if w[0] != t[0] or abs(len(w) - len(t)) > 2:
            continue
        sc = sim(t, w)
        if sc > bs:
            best, bs = w, sc
    thr = 86.0 if len(t) == 5 else 80.0
    return best if (best is not None and bs >= thr) else t

def clean_text(raw):
    s = unicodedata.normalize("NFKC", str(raw)).lower()
    s = re.sub(r"#\s*(\d+[a-z]?)", r" house no \1 ", s)
    s = re.sub(r"\bh\s*\.?\s*no\b\.?", " house no ", s)
    s = re.sub(r"\b(\d+)\s*(?:st|nd|rd|th)\b", r"\1th", s)
    for w, n in ORD_WORDS.items():
        s = re.sub(rf"\b{w}\b(?=\s+(?:cross|main|road))", f"{n}th", s)
    s = re.sub(r"[.,;:()\[\]{}|\"'`~!?*+=<>_\\]", " ", s)
    s = re.sub(r"(?<![0-9])[-/]|[-/](?![0-9])", " ", s)
    s = re.sub(r"\b(\d+)th\s*(?:x|cr|crs|crss)\b", r"\1th cross", s)
    s = re.sub(r"\b(\d+)th\s*(?:mn|mnn|maiin|mainn)\b", r"\1th main", s)
    s = re.sub(r"\b(?:cr|crs)\b(?=\s+\d)", "cross", s)
    s = re.sub(r"\s+", " ", s).strip()
    out = []
    for t in s.split(" "):
        t = ABBR.get(t, t)
        for u in t.split(" "):
            out.append(repair_token(u))
    return " ".join(out)

def script_fracs(raw):
    n = max(len(str(raw)), 1)
    return len(KANNADA_RE.findall(str(raw))) / n, len(DEVANAGARI_RE.findall(str(raw))) / n


def load_extra():
    p = Path(OUT_DIR) / EXTRA_DICT
    if p.exists():
        with open(p, encoding="utf-8") as f:
            extra = json.load(f)
        print(f"[info] loaded extra dictionary {p}")
        return extra
    return {}

def apply_extra_pre(extra):
    for k, v in extra.get("abbrev", {}).items():
        ABBR[k.lower()] = v.lower()
    for t, terms in extra.get("landmark_terms", {}).items():
        LM_TERMS.setdefault(t, [])
        for term in terms:
            LM_TERMS[t].append(re.escape(" ".join(term.lower().split())))

def compile_landmark_regexes():
    LM_RE.clear()
    for t, terms in LM_TERMS.items():
        LM_RE[t] = re.compile(r"(?<![a-z0-9])(?:%s)(?![a-z0-9])" % "|".join(terms))

def raw_tokens(s):
    return set(re.findall(r"[^\W\d_]+", unicodedata.normalize("NFKC", str(s)).lower()))

def init_nlp(extra):
    apply_extra_pre(extra)
    compile_landmark_regexes()

    towns = load("towns.csv")
    locs = load("localities.csv")
    poi = load("landmarks_poi.csv")

    PROTECT.clear()
    for s in list(towns["town_name"]) + list(locs["locality_name"]):
        PROTECT.update(raw_tokens(s))
    repair_token.cache_clear()

    locs = locs.copy()
    locs["pincode"] = locs["pincode"].astype(str)
    locs["name_clean"] = locs["locality_name"].map(clean_text)
    locs["toks"] = locs["name_clean"].str.split()
    locs["dist_phrase"] = locs["toks"].map(
        lambda ts: " ".join(t for t in ts if t not in GENERIC) or None)
    locs.loc[locs["dist_phrase"] == locs["name_clean"], "dist_phrase"] = None
    locs.loc[locs["dist_phrase"].fillna("").str.len() < 4, "dist_phrase"] = None

    towns = towns.copy()
    towns["name_clean"] = towns["town_name"].map(clean_text)

    pin2town = {}
    for r in locs.itertuples():
        pin2town.setdefault(str(r.pincode), set()).add(r.town_id)

    poi = poi.copy()
    loc_by_town = {t: g for t, g in locs.groupby("town_id")}
    pl = []
    for r in poi.itertuples():
        g = loc_by_town.get(r.town_id)
        if g is None or g.empty:
            pl.append((None, np.nan))
            continue
        d = dist(g["centroid_x"].values, g["centroid_y"].values, r.x, r.y)
        k = int(np.argmin(d))
        pl.append((g["locality_id"].iloc[k], float(d[k])))
    poi["locality_id"] = [a for a, _ in pl]
    poi["poi_to_centroid_m"] = [b for _, b in pl]

    poi_sorted = poi.dropna(subset=["locality_id"]).sort_values("poi_to_centroid_m")
    dup = poi_sorted.duplicated(["locality_id", "landmark_type"]).sum()
    poi_sorted = poi_sorted.drop_duplicates(["locality_id", "landmark_type"])
    poi_lookup = {(r.locality_id, r.landmark_type): (r.poi_id, r.x, r.y) for r in poi_sorted.itertuples()}

    types_in_poi = set(poi["landmark_type"].unique())
    types_in_dict = set(LM_TERMS)
    if types_in_poi != types_in_dict:
        print("[warn] landmark_type names differ between landmarks_poi.csv and the dictionary")

    alias = {}
    for lid, phrases in extra.get("locality_aliases", {}).items():
        for ph in phrases:
            alias[clean_text(ph)] = lid

    R.update(dict(towns=towns, locs=locs, poi=poi, pin2town=pin2town, poi_lookup=poi_lookup,
                  alias=alias, poi_dup=int(dup)))
    R["town_names"] = {r.town_id: r.name_clean for r in towns.itertuples()}
    R["loc_centroid"] = {r.locality_id: (r.centroid_x, r.centroid_y, r.town_id) for r in locs.itertuples()}
    R["known_tokens"] = (set(PROTECT) | VOCAB_SET | set(ABBR) | set(ABBR.values()) | GENERIC
                         | set(" ".join(REL_WORDS).split()) | TYPE_HEAD_TOKENS)


def resolve_town(raw, text, town_in=None):
    m = re.search(r"(?<!\d)(\d{6})(?!\d)", str(raw))
    pin = m.group(1) if m else None
    if town_in is not None and pd.notna(town_in):
        return str(town_in), "column", pin
    if pin and len(R["pin2town"].get(pin, ())) == 1:
        return next(iter(R["pin2town"][pin])), "pincode", pin
    hits = {tid for tid, nm in R["town_names"].items() if re.search(PH(nm), text)}
    if len(hits) == 1:
        return next(iter(hits)), "town_name", pin
    return None, "unresolved", pin

def ctx_blocked(text, s, e):
    before = text[:s].split()[-1:]
    after = text[e:].split()[:1]
    if before and before[0] in REL_PRE_TOKENS:
        return True
    if after and after[0] in TYPE_HEAD_TOKENS:
        return True
    return False

def resolve_locality(text, town_id):
    L = R["locs"]
    cands = L if town_id is None else L[L["town_id"] == town_id]
    if cands.empty:
        cands = L
    cand_ids = set(cands["locality_id"])

    for ph, lid in R["alias"].items():
        if lid in cand_ids:
            m = re.search(PH(ph), text)
            if m:
                return lid, "alias_dict", 0.95, m.span()

    hits = []
    for r in cands.itertuples():
        m = re.search(PH(r.name_clean), text)
        if m:
            hits.append((r.locality_id, m.start(), m.end()))
    if hits:
        hits = [h for h in hits if not any(o is not h and o[1] <= h[1] and h[2] <= o[2]
                                           and (o[2] - o[1]) > (h[2] - h[1]) for o in hits)]
        ids = {h[0] for h in hits}
        if len(ids) == 1:
            h = hits[0]
            return h[0], "exact", 1.0, (h[1], h[2])
        return None, "ambiguous_exact", 0.0, None

    dp_count = cands["dist_phrase"].value_counts()
    hits = []
    for r in cands.itertuples():
        dp = r.dist_phrase
        if dp is None or (isinstance(dp, float) and np.isnan(dp)) or dp_count.get(dp, 0) != 1:
            continue
        m = re.search(PH(dp), text)
        if m and not ctx_blocked(text, m.start(), m.end()):
            hits.append((r.locality_id, m.start(), m.end()))
    if len({h[0] for h in hits}) == 1:
        h = hits[0]
        return h[0], "distinctive", 0.8, (h[1], h[2])

    spans = [(m.start(), m.end(), m.group()) for m in re.finditer(r"\S+", text)]
    words = [w for _, _, w in spans]
    best = {}
    for r in cands.itertuples():
        n = len(r.toks)
        for ln in {max(1, n - 1), n, n + 1}:
            if ln > 4:
                continue
            for i in range(len(words) - ln + 1):
                win = words[i:i + ln]
                if any(w.isdigit() for w in win) or all(w in GENERIC for w in win):
                    continue
                wt = " ".join(win)
                if len(wt) < 5 or wt[0] != r.name_clean[0]:
                    continue
                s_, e_ = spans[i][0], spans[i + ln - 1][1]
                if ctx_blocked(text, s_, e_):
                    continue
                sc = sim(wt, r.name_clean)
                if sc > best.get(r.locality_id, (0.0,))[0]:
                    best[r.locality_id] = (sc, s_, e_)
    if best:
        ranked = sorted(best.items(), key=lambda kv: -kv[1][0])
        top_id, (top_sc, s_, e_) = ranked[0]
        second = ranked[1][1][0] if len(ranked) > 1 else 0.0
        if top_sc >= FUZZY_ACCEPT:
            if top_sc - second >= FUZZY_MARGIN:
                return top_id, "fuzzy", round(top_sc / 100.0, 3), (s_, e_)
            return None, "fuzzy_ambiguous", 0.0, None
    return None, "none", 0.0, None

def detect_landmarks(text, mask):
    hits = []
    for t, rx in LM_RE.items():
        for m in rx.finditer(text):
            s, e = m.span()
            if mask is not None and not (e <= mask[0] or s >= mask[1]):
                continue  
            before = text[max(0, s - 24):s]
            after = text[e:e + 16]
            rel = bool(REL_RE.search(before) or REL_RE.search(after))
            hits.append((s, e, t, rel))
    hits.sort()
    return hits

RE_CROSS = [re.compile(r"\b(\d{1,3})th cross\b"), re.compile(r"\bcross (?:no )?(\d{1,3})\b")]
RE_MAIN = [re.compile(r"\b(\d{1,3})th main\b"), re.compile(r"\bmain (?:no )?(\d{1,3})\b")]
RE_GALI = [re.compile(r"\bgall?i (?:no |number )?(\d{1,3})\b")]
RE_ROAD = [re.compile(r"\b(\d{1,3})th road\b"), re.compile(r"\broad (?:no )?(\d{1,3})\b")]
RE_WARD = [re.compile(r"\bward (?:no )?(\d{1,3})\b")]
RE_BLOCK = [re.compile(r"\b([a-z]) block\b"), re.compile(r"\bblock (?:no )?([a-z]|\d{1,2})(?= |$)")]
RE_HOUSE = [re.compile(r"\bhouse (?:no )?([a-z]?\d+[a-z]?(?:[/-]\d+)?)(?= |$)"),
            re.compile(r"\b(?:door|plot|flat) (?:no )?([a-z]?\d+[a-z]?)(?= |$)")]
RE_LEAD_NUM = re.compile(r"^(\d{1,4}[a-z]?(?:[/-]\d+)?)(?= )")
NOT_HOUSE_NEXT = {"cross", "main", "gali", "road", "ward", "block", "floor", "th"}
RE_VILLAGE = re.compile(r"\b(?:village|gram|taluk|tq|district|dist)\b")

def first_match(patterns, text):
    for rx in patterns:
        m = rx.search(text)
        if m:
            return m.group(1)
    return None

def parse_house(text):
    v = first_match(RE_HOUSE, text)
    if v:
        return v
    m = RE_LEAD_NUM.match(text)
    if m:
        nxt = text[m.end():].split()[:1]
        if not (nxt and nxt[0] in NOT_HOUSE_NEXT):
            return m.group(1)
    return None

def parse_addresses(addr, text_col, type_col, town_col):
    rows = []
    types = addr[type_col].values if type_col else [None] * len(addr)
    towns_in = addr[town_col].values if town_col else [None] * len(addr)
    for aid, raw, atype, tin in zip(addr["address_id"].values, addr[text_col].values, types, towns_in):
        text = clean_text(raw)
        kn, dv = script_fracs(raw)
        town_id, town_method, pin = resolve_town(raw, text, tin)
        lid, lmethod, lconf, lspan = resolve_locality(text, town_id)
        if town_id is None and lid is not None:
            town_id, town_method = R["loc_centroid"][lid][2], "via_locality"

        hits = detect_landmarks(text, lspan)
        types_all, types_hc = [], []
        for _, _, t, rel in hits:
            if t not in types_all:
                types_all.append(t)
            if rel and t not in types_hc:
                types_hc.append(t)
        primary = types_hc[0] if types_hc else (types_all[0] if types_all else None)
        lm_conf = "high" if types_hc else ("low" if types_all else None)
        rel_no_type = bool(REL_RE.search(text)) and not types_all
        rel_ctx = None
        if rel_no_type:
            m = REL_RE.search(text)
            rel_ctx = text[max(0, m.start() - 25): m.end() + 25]

        poi = R["poi_lookup"].get((lid, primary)) if (lid is not None and primary) else None

        rec = dict(
            address_id=aid, address_type=atype, text_raw=raw, text_clean=text,
            script_kn_frac=round(kn, 3), script_dev_frac=round(dv, 3), pincode=pin,
            town_id=town_id, town_method=town_method,
            locality_id=lid, locality_method=lmethod, locality_conf=lconf,
            house_no=parse_house(text), cross_no=first_match(RE_CROSS, text), main_no=first_match(RE_MAIN, text),
            gali_no=first_match(RE_GALI, text), road_no=first_match(RE_ROAD, text),
            ward_no=first_match(RE_WARD, text), block=first_match(RE_BLOCK, text),
            is_village_style=bool(RE_VILLAGE.search(text)),
            lm_types_all="|".join(types_all) or None, lm_types_hc="|".join(types_hc) or None,
            lm_primary=primary, lm_conf=lm_conf, lm_ambiguous=len(types_hc) > 1,
            rel_no_type=rel_no_type, rel_context=rel_ctx,
            poi_id=poi[0] if poi else None, poi_x=poi[1] if poi else np.nan, poi_y=poi[2] if poi else np.nan,
        )
        rec["n_fields_found"] = sum(rec[k] is not None and not (isinstance(rec[k], float) and np.isnan(rec[k]))
                                    for k in ["locality_id", "house_no", "cross_no", "main_no", "gali_no",
                                              "road_no", "ward_no", "block", "lm_primary"])
        rows.append(rec)
    return pd.DataFrame(rows)

def print_parse_summary(P):
    hdr("P1-P4 PARSER COVERAGE (all addresses)")
    print(f"fuzzy backend: {FUZZ_BACKEND}   |   rows: {len(P)}   |   POI (locality,type) duplicates: {R['poi_dup']}")
    print("\nTown resolution method:")
    print(P["town_method"].value_counts(dropna=False).to_string())
    print("\nLocality resolution method:")
    print(P["locality_method"].value_counts(dropna=False).to_string())
    if P["address_type"].notna().any():
        print("\nLocality resolved share by address_type:")
        t = P.groupby("address_type").agg(n=("address_id", "size"),
                                          resolved=("locality_id", lambda s: s.notna().mean()))
        print(t.round(3).to_string())
    print("\nLocality resolved share by town:")
    t = P.groupby(P["town_id"].fillna("unknown")).agg(n=("address_id", "size"),
                                                      resolved=("locality_id", lambda s: s.notna().mean()))
    print(t.round(3).to_string())
    print("\nStreet-field coverage (share of addresses with the field), by town:")
    f = P.assign(**{c: P[c].notna() for c in ["house_no", "cross_no", "main_no", "gali_no", "road_no", "ward_no", "block"]})
    print(f.groupby(P["town_id"].fillna("unknown"))[["house_no", "cross_no", "main_no", "gali_no", "road_no",
                                                     "ward_no", "block"]].mean().round(3).to_string())
    print("\nLandmark detection:")
    print(f"  any landmark type found      : {P['lm_types_all'].notna().sum()}")
    print(f"  high confidence (relation word nearby): {P['lm_types_hc'].notna().sum()}")
    print(f"  low confidence (type only)   : {(P['lm_conf'] == 'low').sum()}")
    print(f"  ambiguous (>1 high-conf type): {int(P['lm_ambiguous'].sum())}")
    print(f"  relation word but NO type    : {int(P['rel_no_type'].sum())}   (candidates for the LLM dictionary pass)")
    print(f"  resolved to a POI            : {P['poi_id'].notna().sum()}")
    print("\nLandmark type counts (all / high-confidence):")
    ca, ch = Counter(), Counter()
    for v in P["lm_types_all"].dropna():
        ca.update(v.split("|"))
    for v in P["lm_types_hc"].dropna():
        ch.update(v.split("|"))
    print(pd.DataFrame({"all": pd.Series(ca), "high_conf": pd.Series(ch)}).fillna(0).astype(int)
          .sort_values("all", ascending=False).to_string())


def trail_features(gps):
    g = gps.sort_values(["visit_id", "seq"]).copy()
    g["step"] = np.hypot(g.groupby("visit_id")["x"].diff(), g.groupby("visit_id")["y"].diff())
    a = g.groupby("visit_id").agg(n_points=("seq", "size"), trail_len=("step", "sum"),
                                  x0=("x", "first"), y0=("y", "first"), x1=("x", "last"), y1=("y", "last"),
                                  t0=("point_ts", "first"), t1=("point_ts", "last")).reset_index()
    a["net_disp"] = np.hypot(a["x1"] - a["x0"], a["y1"] - a["y0"])
    a["path_eff"] = np.where(a["trail_len"] > 0, a["net_disp"] / a["trail_len"].replace(0, np.nan), np.nan)
    if pd.api.types.is_numeric_dtype(a["t0"]):
        a["trail_dur_s"] = a["t1"] - a["t0"]
    else:
        a["trail_dur_s"] = (pd.to_datetime(a["t1"], errors="coerce") - pd.to_datetime(a["t0"], errors="coerce")).dt.total_seconds()
    return a.drop(columns=["t0", "t1"])

def integrity_signals(fv, gps, base):
    from sklearn.cluster import DBSCAN

    tr = trail_features(gps)
    fv = fv.merge(tr, on="visit_id", how="left")
    b = base.rename(columns={"precision": "base_precision"})[["address_id", "geocoder_x", "geocoder_y", "base_precision"]]
    fv = fv.merge(b, on="address_id", how="left")

    fv["checkin_to_base_m"] = dist(fv["checkin_x"], fv["checkin_y"], fv["geocoder_x"], fv["geocoder_y"])
    fv["start_to_base_m"] = dist(fv["x0"], fv["y0"], fv["geocoder_x"], fv["geocoder_y"])
    fv["checkin_to_trail_end_m"] = dist(fv["checkin_x"], fv["checkin_y"], fv["x1"], fv["y1"])

    labels = np.full(len(fv), -1)
    valid = fv[["checkin_x", "checkin_y"]].notna().all(axis=1).values
    labels[valid] = DBSCAN(eps=DBSCAN_EPS_M, min_samples=DBSCAN_MIN_SAMPLES).fit_predict(
        fv.loc[valid, ["checkin_x", "checkin_y"]].values)
    fv["cl"] = labels

    rep = []
    for c in sorted(set(labels) - {-1}):
        sub = fv[fv["cl"] == c]
        u = sub.drop_duplicates("address_id")[["geocoder_x", "geocoder_y"]].dropna()
        if len(u):
            cx, cy = u["geocoder_x"].mean(), u["geocoder_y"].mean()
            spread = float(np.sqrt(np.mean((u["geocoder_x"] - cx) ** 2 + (u["geocoder_y"] - cy) ** 2)))
        else:
            spread = np.nan
        top = sub["agent_id"].value_counts()
        rep.append(dict(cluster=c, visits=len(sub), addresses=sub["address_id"].nunique(),
                        pin_spread_m=round(spread, 1), centre_x=round(sub["checkin_x"].mean(), 1),
                        centre_y=round(sub["checkin_y"].mean(), 1), n_agents=sub["agent_id"].nunique(),
                        top_agent=top.index[0], top_agent_share=round(top.iloc[0] / len(sub), 3)))
    cl_rep = pd.DataFrame(rep)
    hub_cl = set(cl_rep.loc[(cl_rep["visits"] >= HUB_MIN_VISITS) & (cl_rep["pin_spread_m"] >= HUB_MIN_SPREAD_M), "cluster"]) \
        if len(cl_rep) else set()
    cl_rep["is_hub_cluster"] = cl_rep["cluster"].isin(hub_cl) if len(cl_rep) else []
    spread_map = dict(zip(cl_rep["cluster"], cl_rep["pin_spread_m"])) if len(cl_rep) else {}
    fv["cl_pin_spread_m"] = fv["cl"].map(spread_map)

    fv["photo_addrs"] = fv.groupby("photo_hash")["address_id"].transform("nunique").fillna(1)
    fv["photo_reuse_n"] = fv.groupby("photo_hash")["visit_id"].transform("size").fillna(1)

    fv["_ts"] = pd.to_datetime(fv["checkin_ts"], errors="coerce")
    o = fv.sort_values(["agent_id", "_ts"])
    gb = o.groupby("agent_id")
    dt_h = gb["_ts"].diff().dt.total_seconds() / 3600.0
    dd_km = np.hypot(gb["checkin_x"].diff(), gb["checkin_y"].diff()) / 1000.0
    same_day = o["_ts"].dt.date == gb["_ts"].shift().dt.date
    fv["implied_speed_kmh"] = np.where((dt_h > 0) & same_day, dd_km / dt_h, np.nan)
    fv = fv.drop(columns=["_ts"])

    fv["hub_member"] = fv["cl"].isin(hub_cl)
    fv["tiny_trail"] = (fv["trail_len"] < 200) & (fv["start_to_base_m"] > 1500)
    fv["is_hub2"] = fv["hub_member"] | (fv["photo_addrs"] >= PHOTO_ADDR_THRESHOLD)
    fv["integrity_risk"] = (RISK_W["hub"] * fv["hub_member"].astype(float)
                            + RISK_W["photo"] * np.clip((fv["photo_addrs"] - 1) / 2.0, 0, 1)
                            + RISK_W["tiny"] * fv["tiny_trail"].astype(float)
                            + RISK_W["start"] * np.clip((fv["start_to_base_m"].fillna(0) - 500) / 1500.0, 0, 1))
    fv["tier"] = np.where(fv["is_hub2"], "flagged",
                          np.where(fv["outcome"] == "address_not_traceable", "not_traceable", "clean"))

    resid = np.full(len(fv), np.nan)
    pos = {idx: i for i, idx in enumerate(fv.index)}
    for _, g in fv.groupby("address_id"):
        c = g[g["tier"] == "clean"]
        if c.empty:
            continue
        for idx, row in g.iterrows():
            others = c[c["agent_id"] != row["agent_id"]]
            if len(others):
                resid[pos[idx]] = float(dist(row["checkin_x"], row["checkin_y"],
                                             others["checkin_x"].median(), others["checkin_y"].median()))
    fv["lao_resid_m"] = resid
    return fv, cl_rep, hub_cl

def build_pseudo_truth(fv, addr, splits):
    clean = fv[fv["tier"] == "clean"].copy()
    pt = clean.groupby("address_id").agg(pt_x=("checkin_x", "median"), pt_y=("checkin_y", "median"),
                                         pt_k=("visit_id", "size")).reset_index()
    clean = clean.merge(pt[["address_id", "pt_x", "pt_y"]], on="address_id")
    clean["_d"] = dist(clean["checkin_x"], clean["checkin_y"], clean["pt_x"], clean["pt_y"])
    pt = pt.merge(clean.groupby("address_id")["_d"].median().rename("pt_spread_m").reset_index(), on="address_id")
    acct = addr[["address_id", "account_id"]] if "account_id" in addr.columns else \
        fv[["address_id", "account_id"]].drop_duplicates("address_id")
    sp_col = pick_col(splits, ["split", "set", "subset", "partition"], "split")
    pt = pt.merge(acct, on="address_id", how="left").merge(
        splits[["account_id", sp_col]].rename(columns={sp_col: "split"}), on="account_id", how="left")
    return pt


def norm_num(v):
    v = str(v).strip().lower()
    if v in ("", "nan", "none", "na", "-"):
        return None
    v = re.sub(r"(?<=\d)(st|nd|rd|th)$", "", v)
    v = re.sub(r"\.0$", "", v)
    if v.isdigit():
        v = v.lstrip("0") or "0"
    return v

def norm_set(v):
    v = str(v).strip().lower()
    if v in ("", "nan", "none"):
        return set()
    return {x.strip().replace(" ", "_") for x in v.split("|") if x.strip()}

def evaluate_parser(P, half):
    p = Path(OUT_DIR) / LABELS_FILLED
    if not p.exists():
        print(f"\n[info] {p} not found; skipping parser evaluation (label the sample first).")
        return
    lab = pd.read_csv(p, dtype=str, keep_default_na=False)
    if "split_half" not in lab.columns:
        print("[warn] split_half column missing in the filled sheet")
        return
    lab = lab[lab["split_half"] == half].copy()
    hdr(f"PARSER EVALUATION on the {half.upper()} half (n={len(lab)})")
    if half == "test":
        print("!! You are scoring the LOCKED test half. Do not tune the parser on this afterwards.")
    m = lab.merge(P, on="address_id", how="left", suffixes=("_lab", ""))
    errors = []
    out = []

    for name, lcol, pcol in [("town", "label_town_id", "town_id"), ("locality", "label_locality_id", "locality_id")]:
        L = m[lcol].str.strip().str.upper().replace("", np.nan)
        Pp = m[pcol].astype(object).where(m[pcol].notna(), np.nan)
        Pp = Pp.map(lambda v: str(v).strip().upper() if pd.notna(v) else np.nan)
        n = len(m)
        n_lab, n_pred = int(L.notna().sum()), int(Pp.notna().sum())
        tp = int(((L == Pp) & L.notna() & Pp.notna()).sum())
        out.append(dict(field=name, n=n, labelled=n_lab, predicted=n_pred, correct=tp,
                        precision=tp / n_pred if n_pred else np.nan, recall=tp / n_lab if n_lab else np.nan))
        for i in m.index[(L != Pp) & ~(L.isna() & Pp.isna())]:
            errors.append(dict(address_id=m.at[i, "address_id"], field=name, label=L[i], pred=Pp[i],
                               text_clean=m.at[i, "text_clean"]))

    for name, lcol, pcol in [("house_no", "label_house_no", "house_no"), ("cross", "label_cross", "cross_no"),
                             ("main", "label_main", "main_no"), ("gali", "label_gali", "gali_no"),
                             ("road", "label_road", "road_no"), ("ward", "label_ward", "ward_no"),
                             ("block", "label_block", "block")]:
        L = m[lcol].map(norm_num)
        Pp = m[pcol].map(lambda v: norm_num(v) if pd.notna(v) else None)
        n_lab, n_pred = int(L.notna().sum()), int(Pp.notna().sum())
        tp = int(sum(1 for a, b in zip(L, Pp) if a is not None and a == b))
        out.append(dict(field=name, n=len(m), labelled=n_lab, predicted=n_pred, correct=tp,
                        precision=tp / n_pred if n_pred else np.nan, recall=tp / n_lab if n_lab else np.nan))
        for i, (a, b) in zip(m.index, zip(L, Pp)):
            if a != b:
                errors.append(dict(address_id=m.at[i, "address_id"], field=name, label=a, pred=b,
                                   text_clean=m.at[i, "text_clean"]))

    for name, pcol in [("landmark_all", "lm_types_all"), ("landmark_hc", "lm_types_hc")]:
        tp = fp = fn = 0
        for i in m.index:
            L = norm_set(m.at[i, "label_landmark_types"])
            Pp = norm_set(m.at[i, pcol] if pd.notna(m.at[i, pcol]) else "")
            tp += len(L & Pp)
            fp += len(Pp - L)
            fn += len(L - Pp)
            if L != Pp:
                errors.append(dict(address_id=m.at[i, "address_id"], field=name, label="|".join(sorted(L)),
                                   pred="|".join(sorted(Pp)), text_clean=m.at[i, "text_clean"]))
        out.append(dict(field=name, n=len(m), labelled=tp + fn, predicted=tp + fp, correct=tp,
                        precision=tp / (tp + fp) if tp + fp else np.nan, recall=tp / (tp + fn) if tp + fn else np.nan))

    res = pd.DataFrame(out)
    print(res.round(3).to_string(index=False))
    res.to_csv(Path(OUT_DIR) / f"parser_eval_{half}.csv", index=False)
    pd.DataFrame(errors).to_csv(Path(OUT_DIR) / f"parser_errors_{half}.csv", index=False)
    print(f"\nsaved parser_eval_{half}.csv and parser_errors_{half}.csv (every disagreement, with the cleaned text)")


def run_parse():
    out = Path(OUT_DIR)
    out.mkdir(parents=True, exist_ok=True)
    extra = load_extra()
    validate_all_datasets_present()
    init_nlp(extra)

    accounts = load("accounts.csv")
    agents = load("agents.csv")
    lenders = load("lenders.csv")
    
    addr = load("addresses.csv")
    text_col = detect_text_col(addr)
    type_col = pick_col(addr, ["address_type", "type"], "address type", required=False)
    town_col = pick_col(addr, ["town_id"], "town", required=False)
    print(f"[info] addresses columns: {list(addr.columns)} | text column = '{text_col}'")

    P = parse_addresses(addr, text_col, type_col, town_col)
    if "account_id" in addr.columns:
        P.insert(1, "account_id", addr["account_id"].values)
    print_parse_summary(P)

    resid_loc = P[P["locality_id"].isna()][["address_id", "address_type", "town_id", "text_clean"]]
    resid_loc.to_csv(out / "residual_locality.csv", index=False, encoding="utf-8-sig")
    resid_lm = P[P["rel_no_type"]][["address_id", "text_clean", "rel_context"]]
    resid_lm.to_csv(out / "residual_landmark.csv", index=False, encoding="utf-8-sig")
    known = R["known_tokens"]
    tok = Counter()
    for t in pd.concat([resid_loc["text_clean"], resid_lm["text_clean"]]).unique():
        for w in t.split():
            if not w.isdigit() and len(w) >= 3 and w not in known:
                tok[w] += 1
    pd.DataFrame(tok.most_common(300), columns=["token", "n_addresses"]).to_csv(
        out / "unmatched_tokens.csv", index=False, encoding="utf-8-sig")
    print(f"\nresidual files: residual_locality.csv ({len(resid_loc)}), residual_landmark.csv ({len(resid_lm)}), unmatched_tokens.csv")

    hdr("INTEGRITY SIGNALS")
    fv = load("field_visits.csv")
    gps = load("visit_gps_points.csv")
    base = load("baseline_geocodes.csv")
    splits = load("splits.csv")
    fv, cl_rep, hub_cl = integrity_signals(fv, gps, base)

    print("DBSCAN clusters (legit ones have a small pin spread; hubs have a large one):")
    print(cl_rep.sort_values("visits", ascending=False).to_string(index=False))
    print(f"\nhub clusters used: {sorted(hub_cl)}")
    print(f"check-in vs trail end (m): median {np.nanmedian(fv['checkin_to_trail_end_m']):.1f}, "
          f"p75 {np.nanpercentile(fv['checkin_to_trail_end_m'], 75):.1f}, max {np.nanmax(fv['checkin_to_trail_end_m']):.1f}   (expect ~15.9 / 28.4 / 155.7)")
    print("\nTier counts:")
    print(fv["tier"].value_counts().to_string())
    ag = fv.groupby("agent_id").agg(
        visits=("visit_id", "size"), flagged=("is_hub2", "sum"), flagged_share=("is_hub2", "mean"),
        hub_member=("hub_member", "sum"), photo_flag=("photo_addrs", lambda s: int((s >= PHOTO_ADDR_THRESHOLD).sum())),
        tiny_trail=("tiny_trail", "sum"), mean_risk=("integrity_risk", "mean"),
        lao_resid_median_clean=("lao_resid_m", lambda s: np.nan))
    clean_res = fv[fv["tier"] == "clean"].groupby("agent_id")["lao_resid_m"].agg(
        lao_resid_median_clean="median", lao_resid_p90_clean=lambda s: s.quantile(0.9))
    ag = ag.drop(columns=["lao_resid_median_clean"]).join(clean_res)
    print("\nPer-agent integrity report (expect FA009 ~231 flagged; the others near zero):")
    print(ag.round(3).to_string())
    print("\nFlagged visits outside FA009 (inspect these by hand):")
    cols = ["visit_id", "agent_id", "address_id", "outcome", "cl", "photo_addrs", "trail_len", "start_to_base_m", "integrity_risk"]
    print(fv[fv["is_hub2"] & (fv["agent_id"] != "FA009")][cols].round(1).to_string(index=False))
    ag.reset_index().to_csv(out / "agent_integrity_report.csv", index=False)
    cl_rep.to_csv(out / "integrity_cluster_report.csv", index=False)

    pt = build_pseudo_truth(fv, addr, splits)
    surv = load("surveyed_addresses.csv")
    hdr("PSEUDO-TRUTH SANITY (against surveyed truth, reporting only)")
    j = pt.merge(surv, on="address_id")
    print(f"addresses with pseudo-truth: {len(pt)}  (by split: {pt['split'].value_counts().to_dict()})")
    print("pseudo-truth k distribution:", pt["pt_k"].value_counts().sort_index().to_dict())
    s = err_stats(dist(j["pt_x"], j["pt_y"], j["surveyed_x"], j["surveyed_y"]))
    print(f"pseudo-truth vs surveyed on {s['n']} overlapping addresses: median {s['median']:.1f} m, "
          f"p90 {s['p90']:.1f} m, max {s['max']:.1f} m   (earlier EDA: 14 / 68 / 252)")
    jv = fv.merge(surv, on="address_id")
    jv["err_to_truth"] = dist(jv["checkin_x"], jv["checkin_y"], jv["surveyed_x"], jv["surveyed_y"])
    rows = []
    for tier, g in jv.groupby("tier"):
        e = err_stats(g["err_to_truth"])
        rows.append(dict(tier=tier, **e))
    print("\nVisit check-in error vs surveyed truth by tier (earlier EDA: clean 24/129, flagged 5368/7829, not_traceable 681/2349):")
    print(pd.DataFrame(rows).round(1).to_string(index=False))

    hdr("LOCALITY ASSIGNMENT SANITY (rank of the assigned locality by distance to pseudo-truth)")
    L = R["locs"]
    m = P.merge(pt[["address_id", "pt_x", "pt_y", "pt_k"]], on="address_id")
    m = m[m["locality_id"].notna() & m["town_id"].notna()]
    ranks, d_assigned = [], []
    for r in m.itertuples():
        g = L[L["town_id"] == r.town_id]
        d = dist(g["centroid_x"].values, g["centroid_y"].values, r.pt_x, r.pt_y)
        order = g["locality_id"].values[np.argsort(d)]
        if r.locality_id in order:
            ranks.append(int(np.where(order == r.locality_id)[0][0]) + 1)
            cx, cy, _ = R["loc_centroid"][r.locality_id]
            d_assigned.append(float(dist(cx, cy, r.pt_x, r.pt_y)))
        else:
            ranks.append(np.nan)
            d_assigned.append(np.nan)
    m = m.assign(loc_rank=ranks, dist_to_assigned_centroid_m=d_assigned)
    print(f"addresses checked: {len(m)}")
    if len(m):
        print("rank of assigned locality (1 = nearest centroid to the pseudo-truth):")
        print(m["loc_rank"].value_counts(normalize=True).sort_index().round(3).head(8).to_string())
        print("\nby method: share with rank <= 3, and median distance to the assigned centroid (m)")
        print(m.groupby("locality_method").agg(n=("address_id", "size"), rank_le3=("loc_rank", lambda s: (s <= 3).mean()),
                                               med_dist=("dist_to_assigned_centroid_m", "median")).round(3).to_string())
        m[m["loc_rank"] > 3][["address_id", "text_clean", "locality_id", "locality_method", "loc_rank"]].to_csv(
            out / "locality_suspects.csv", index=False, encoding="utf-8-sig")
        print("saved locality_suspects.csv (assigned locality ranked > 3: look at these by eye)")

    P.to_csv(out / "address_parsed.csv", index=False, encoding="utf-8-sig")
    keep = ["visit_id", "account_id", "address_id", "agent_id", "outcome", "checkin_x", "checkin_y", "gps_accuracy_m",
            "dwell_s", "n_points", "trail_len", "net_disp", "path_eff", "trail_dur_s", "start_to_base_m",
            "checkin_to_base_m", "checkin_to_trail_end_m", "base_precision", "cl", "cl_pin_spread_m", "hub_member",
            "photo_addrs", "photo_reuse_n", "tiny_trail", "implied_speed_kmh", "is_hub2", "integrity_risk",
            "tier", "lao_resid_m"]
    fv[[c for c in keep if c in fv.columns]].to_csv(out / "visit_features.csv", index=False)
    ev = P[["address_id", "town_id", "locality_id", "locality_method", "lm_primary", "poi_id", "poi_x", "poi_y"]] \
        .merge(base.rename(columns={"precision": "base_precision"}), on="address_id", how="left") \
        .merge(pt, on="address_id", how="left")
    ev["is_surveyed"] = ev["address_id"].isin(surv["address_id"])
    ev.to_csv(out / "address_evidence.csv", index=False)
    hdr("FILES WRITTEN")
    for f in ["address_parsed.csv", "visit_features.csv", "address_evidence.csv", "integrity_cluster_report.csv",
              "agent_integrity_report.csv", "residual_locality.csv", "residual_landmark.csv", "unmatched_tokens.csv",
              "locality_suspects.csv"]:
        print(" ", out / f)

    evaluate_parser(P, EVAL_HALF)

def main(argv=None):
    import argparse
    ap=argparse.ArgumentParser()
    ap.add_argument("--data-dir", required=True)
    ap.add_argument("--out-dir", default="ps3_out")
    args=ap.parse_args(argv)
    global DATA_DIR, OUT_DIR
    DATA_DIR=args.data_dir
    OUT_DIR=args.out_dir
    run_parse()

if __name__ == "__main__":
    main()


In [ ]:
%%writefile cn_build_master.py
"""Combine Stage 1's parsed addresses, GPS evidence, and visit reliability."""
from pathlib import Path
import numpy as np
import pandas as pd


def build_master(work_dir=Path('.')):
    work_dir=Path(work_dir)
    parsed=pd.read_csv(work_dir/'ps3_out'/'address_parsed.csv')
    evidence=pd.read_csv(work_dir/'ps3_out'/'address_evidence.csv')
    visits=pd.read_csv(work_dir/'ps3_out'/'visit_features.csv')
    needed=['address_id','geocoder_x','geocoder_y','base_precision','is_surveyed','pt_k','pt_spread_m','pt_x','pt_y','split']
    if not parsed.address_id.is_unique or not evidence.address_id.is_unique:
        raise ValueError('Source address IDs are not unique')
    master=parsed.merge(evidence[needed], on='address_id', how='left', validate='one_to_one')
    grouped=visits.groupby('address_id',dropna=False).agg(
        total_visits=('visit_id','size'),
        clean_visits=('tier',lambda x:(x=='clean').sum()),
        flagged_visits=('tier',lambda x:(x=='flagged').sum()),
        avg_integrity_risk=('integrity_risk','mean'),
        agents_involved=('agent_id','nunique'),
    ).reset_index()
    master=master.merge(grouped,on='address_id',how='left',validate='one_to_one')
    for c in ['total_visits','clean_visits','flagged_visits','agents_involved']:
        master[c]=master[c].fillna(0).astype(int)
    if len(master)!=len(parsed) or master.address_id.isna().any():
        raise AssertionError('Address merge changed cardinality')
    target=work_dir/'ps3_out'/'ps3_master_processed.csv'
    master.to_csv(target,index=False)
    return master

if __name__=='__main__':
    print('Built master:',build_master().shape)


In [ ]:
run_script('cn_stage1_preprocess.py', '--data-dir',raw_dir,'--out-dir','ps3_out')
run_script('cn_build_master.py')
master = pd.read_csv('ps3_out/ps3_master_processed.csv')
assert len(master) == 3117 and master.address_id.is_unique
print('Master assembled directly from raw data:', master.shape)
print('Clean visits:', int(master.clean_visits.sum()), '| Resolved raw POIs:',int(master.poi_id.notna().sum()))

## 2. Architecture 5.2 — corrected geographic pin

Visited addresses are estimated with reliability-weighted visit geometry; unvisited addresses use structured street, locality, POI, machine-learning, and commercial-baseline evidence. The original OOD-aware expert router and **its full cross-validation/training procedure** are retained. Surveyed coordinates are used for the same calibration/evaluation design as the historical model, not as direct features for an address at inference time.


In [ ]:
%%writefile cn_architecture52.py
from __future__ import annotations

import json
import math
import os
import re
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor, Ridge
from sklearn.model_selection import StratifiedKFold, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


MASTER_PATH = "ps3_out/ps3_master_processed.csv"
SURVEY_PATH = "surveyed_addresses.csv"
OPTIONAL_VISIT_PATH = "ps3_out/visit_features.csv"
OUTPUT_DIR = "stage2_arch5"

SEED = 42
N_FOLDS = 5
CONF_LEVEL = 0.90
MIN_MODEL_ROWS = 25


TAIL_P90_WEIGHT = 1.0
TAIL_MEDIAN_WEIGHT = 0.25
TAIL_MEAN_WEIGHT = 0.05

warnings.filterwarnings("ignore", category=RuntimeWarning)
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count() or 4))

OVERRIDE = {
    "master_address_id": None,
    "survey_address_id": None,
    "truth_x": None,
    "truth_y": None,
    "baseline_x": None,
    "baseline_y": None,
    "pseudo_x": None,
    "pseudo_y": None,
    "poi_x": None,
    "poi_y": None,
    "locality_id": None,
    "visit_x": None,
    "visit_y": None,
    "is_hub2": None,
    "integrity_risk": None,
    "outcome": None,
    "survey_split": None,
}

ROLE_CANDIDATES = {
    "address_id": ["address_id", "addr_id", "addressid", "address_key", "addr_key", "record_id"],
    "address_text": ["address", "address_text", "raw_address", "full_address", "customer_address", "text_raw"],
    "truth_x": ["truth_x", "true_x", "survey_x", "surveyed_x", "ground_truth_x", "gt_x", "actual_x", "x_truth", "x_true", "x"],
    "truth_y": ["truth_y", "true_y", "survey_y", "surveyed_y", "ground_truth_y", "gt_y", "actual_y", "y_truth", "y_true", "y"],
    "baseline_x": ["baseline_x", "commercial_x", "geocoder_x", "commercial_geocoder_x", "base_x", "baseline_pin_x", "geocode_x"],
    "baseline_y": ["baseline_y", "commercial_y", "geocoder_y", "commercial_geocoder_y", "base_y", "baseline_pin_y", "geocode_y"],
    "pseudo_x": ["pseudo_x", "pseudo_truth_x", "pseudo_gt_x", "consensus_x", "clean_median_x", "pseudo_pin_x", "pt_x"],
    "pseudo_y": ["pseudo_y", "pseudo_truth_y", "pseudo_gt_y", "consensus_y", "clean_median_y", "pseudo_pin_y", "pt_y"],
    "poi_x": ["poi_x", "matched_poi_x", "landmark_poi_x", "poi_coord_x", "landmark_x"],
    "poi_y": ["poi_y", "matched_poi_y", "landmark_poi_y", "poi_coord_y", "landmark_y"],
    "locality_id": ["locality_id", "locality", "canonical_locality", "matched_locality", "locality_name"],
    "visit_x": ["visit_x", "checkin_x", "check_in_x", "checkin_coord_x", "gps_x", "visit_easting", "checkin_easting", "agent_x"],
    "visit_y": ["visit_y", "checkin_y", "check_in_y", "checkin_coord_y", "gps_y", "visit_northing", "checkin_northing", "agent_y"],
    "is_hub2": ["is_hub2", "hub2", "is_fake_hub", "fake_hub_flag"],
    "integrity_risk": ["integrity_risk", "visit_integrity_risk", "risk_score", "fraud_risk"],
    "outcome": ["outcome", "visit_outcome", "field_outcome", "result"],
    "survey_split": ["split", "survey_split", "dataset_split", "set", "fold_type", "is_test", "is_holdout", "holdout"],
}

STRUCTURED_NUM_CANDIDATES = [
    "house_no", "house_number", "cross_no", "cross_number", "main_no", "main_number",
    "gali_no", "gali_number", "road_no", "road_number", "ward_no", "ward_number"
]
STRUCTURED_CAT_CANDIDATES = [
    "block", "locality_id", "town_id", "lm_primary", "rel_no_type", "rel_context",
    "poi_id", "locality_method", "town_method", "base_precision"
]
EXTRA_NUM_CANDIDATES = [
    "locality_conf", "lm_conf", "n_fields_found", "script_kn_frac", "script_dev_frac",
    "is_village_style"
]


def norm_col(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    return re.sub(r"_+", "_", s).strip("_")


def normalize_df_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [norm_col(c) for c in out.columns]
    return out


def find_col(df: pd.DataFrame, role: str, override_key: Optional[str] = None) -> Optional[str]:
    if override_key and OVERRIDE.get(override_key):
        c = norm_col(OVERRIDE[override_key])
        if c not in df.columns:
            raise KeyError(f"Override {override_key}={c!r} not found. Available columns: {list(df.columns)}")
        return c
    for c in ROLE_CANDIDATES.get(role, []):
        if c in df.columns:
            return c
    return None


def require_col(c: Optional[str], role: str, df: pd.DataFrame) -> str:
    if c is None:
        raise KeyError(f"Could not detect {role}. Available columns:\n{list(df.columns)}")
    return c


def to_bool_series(s: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False)
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce").fillna(0) != 0
    return s.astype(str).str.strip().str.lower().isin({"1", "true", "t", "yes", "y"})


def euclid(x1, y1, x2, y2):
    return np.sqrt((np.asarray(x1, float) - np.asarray(x2, float)) ** 2 +
                   (np.asarray(y1, float) - np.asarray(y2, float)) ** 2)


def finite_xy(x, y) -> bool:
    return np.isfinite(float(x)) and np.isfinite(float(y))


def weighted_geometric_median(points: np.ndarray, weights: np.ndarray, max_iter=300, tol=1e-5):
    P = np.asarray(points, float)
    W = np.asarray(weights, float)
    mask = np.all(np.isfinite(P), axis=1) & np.isfinite(W) & (W > 0)
    P, W = P[mask], W[mask]
    if len(P) == 0:
        return np.array([np.nan, np.nan])
    if len(P) == 1:
        return P[0].copy()
    y = np.average(P, axis=0, weights=W)
    for _ in range(max_iter):
        d = np.linalg.norm(P - y, axis=1)
        hit = d < 1e-10
        if hit.any():
            return P[np.argmax(W * hit)]
        ww = W / np.maximum(d, 1e-9)
        y1 = np.sum(P * ww[:, None], axis=0) / ww.sum()
        if np.linalg.norm(y1 - y) < tol:
            return y1
        y = y1
    return y


def onehot_dense():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def usable_num_cols(df: pd.DataFrame, candidates: Sequence[str]) -> List[str]:
    out = []
    for c in candidates:
        if c not in df.columns:
            continue
        vals = pd.to_numeric(df[c], errors="coerce")
        if vals.notna().any():
            out.append(c)
    return list(dict.fromkeys(out))


def usable_cat_cols(df: pd.DataFrame, candidates: Sequence[str]) -> List[str]:
    out = []
    for c in candidates:
        if c not in df.columns:
            continue
        vals = df[c].replace({"": np.nan})
        if vals.notna().any():
            out.append(c)
    return list(dict.fromkeys(out))


def build_preprocessor(num_cols: List[str], cat_cols: List[str], scale_numeric=False):
    transformers = []
    if num_cols:
        steps = [("impute", SimpleImputer(strategy="median"))]
        if scale_numeric:
            steps.append(("scale", StandardScaler()))
        transformers.append(("num", Pipeline(steps), num_cols))
    if cat_cols:
        transformers.append((
            "cat",
            Pipeline([
                ("impute", SimpleImputer(strategy="most_frequent")),
                ("onehot", onehot_dense()),
            ]),
            cat_cols,
        ))
    if not transformers:
        raise ValueError("No usable model features.")
    return ColumnTransformer(transformers, remainder="drop")


def make_pipeline(kind: str, num_cols: List[str], cat_cols: List[str]):
    if kind == "grid_huber":
        pre = build_preprocessor(num_cols, cat_cols, scale_numeric=True)
        model = HuberRegressor(epsilon=1.35, alpha=1e-4, max_iter=3000, tol=1e-5)
    elif kind == "resid_hgb":
        pre = build_preprocessor(num_cols, cat_cols, scale_numeric=False)
        model = HistGradientBoostingRegressor(
            max_iter=300, learning_rate=0.045, max_leaf_nodes=15,
            min_samples_leaf=12, l2_regularization=1.0, random_state=SEED
        )
    elif kind == "extra":
        pre = build_preprocessor(num_cols, cat_cols, scale_numeric=False)
        model = ExtraTreesRegressor(
            n_estimators=450, min_samples_leaf=3, max_features=0.85,
            random_state=SEED, n_jobs=-1
        )
    elif kind == "quantile":
        pre = build_preprocessor(num_cols, cat_cols, scale_numeric=False)
        model = GradientBoostingRegressor(
            loss="quantile", alpha=CONF_LEVEL, n_estimators=220,
            learning_rate=0.035, max_depth=2, min_samples_leaf=5,
            random_state=SEED
        )
    else:
        raise ValueError(kind)
    return Pipeline([("pre", pre), ("model", model)])


def fit_pair(train: pd.DataFrame, num_cols, cat_cols, tx, ty, kind):
    features = num_cols + cat_cols
    tr = train.loc[train[[tx, ty]].notna().all(axis=1)].copy()
    if len(tr) < MIN_MODEL_ROWS or not features:
        return None, None
    mx = make_pipeline(kind, num_cols, cat_cols)
    my = make_pipeline(kind, num_cols, cat_cols)
    try:
        mx.fit(tr[features], tr[tx])
        my.fit(tr[features], tr[ty])
    except Exception as e:
        # Grid Huber is the only optimization that can occasionally struggle.
        # Fall back to scaled Ridge so the notebook never dies on convergence.
        if kind == "grid_huber":
            pre = build_preprocessor(num_cols, cat_cols, scale_numeric=True)
            mx = Pipeline([("pre", clone(pre)), ("model", Ridge(alpha=10.0))])
            my = Pipeline([("pre", clone(pre)), ("model", Ridge(alpha=10.0))])
            mx.fit(tr[features], tr[tx])
            my.fit(tr[features], tr[ty])
        else:
            raise RuntimeError(f"Failed fitting {kind}: {e}")
    return mx, my


def pair_predict(models, df: pd.DataFrame, features: List[str]) -> np.ndarray:
    out = np.full((len(df), 2), np.nan)
    mx, my = models
    if mx is None or my is None or not features:
        return out
    out[:, 0] = mx.predict(df[features])
    out[:, 1] = my.predict(df[features])
    return out


def conformal_quantile(values: Sequence[float], q=CONF_LEVEL) -> float:
    x = np.sort(np.asarray([v for v in values if np.isfinite(v)], float))
    if len(x) == 0:
        return np.nan
    k = int(math.ceil((len(x) + 1) * q))
    k = min(max(k, 1), len(x))
    return float(x[k - 1])


def summarize(frame: pd.DataFrame, err_col: str, radius_col: Optional[str] = None) -> Dict[str, float]:
    e = pd.to_numeric(frame[err_col], errors="coerce").dropna().to_numpy(float)
    out = {
        "n": int(len(e)),
        "median_error_m": float(np.median(e)) if len(e) else np.nan,
        "p90_error_m": float(np.quantile(e, 0.90)) if len(e) else np.nan,
        "mean_error_m": float(np.mean(e)) if len(e) else np.nan,
        "pct_under_100m": float(np.mean(e <= 100) * 100) if len(e) else np.nan,
        "pct_under_250m": float(np.mean(e <= 250) * 100) if len(e) else np.nan,
        "pct_under_500m": float(np.mean(e <= 500) * 100) if len(e) else np.nan,
    }
    if radius_col and radius_col in frame.columns:
        m = frame[[err_col, radius_col]].notna().all(axis=1)
        if m.any():
            ee = frame.loc[m, err_col].to_numpy(float)
            rr = frame.loc[m, radius_col].to_numpy(float)
            out.update({
                "coverage_pct": float(np.mean(ee <= rr) * 100),
                "median_radius_m": float(np.median(rr)),
                "p90_radius_m": float(np.quantile(rr, 0.90)),
            })
    return out


def build_visit_consensus(addr: pd.DataFrame, master_id: str, px: str, py: str) -> Tuple[pd.DataFrame, str]:
    optional = Path(OPTIONAL_VISIT_PATH)
    if optional.exists():
        try:
            vf = normalize_df_columns(pd.read_csv(optional))
            vmid = find_col(vf, "address_id")
            vx = find_col(vf, "visit_x", "visit_x")
            vy = find_col(vf, "visit_y", "visit_y")
            if vmid and vx and vy:
                vf[vmid] = vf[vmid].astype(str)
                if vmid != master_id:
                    vf = vf.rename(columns={vmid: master_id})
                hub = find_col(vf, "is_hub2", "is_hub2")
                risk = find_col(vf, "integrity_risk", "integrity_risk")
                outcome = find_col(vf, "outcome", "outcome")
                vf[vx] = pd.to_numeric(vf[vx], errors="coerce")
                vf[vy] = pd.to_numeric(vf[vy], errors="coerce")
                v = vf.loc[vf[[vx, vy]].notna().all(axis=1)].copy()
                v["__hub"] = to_bool_series(v[hub]) if hub else False
                if outcome:
                    os_ = v[outcome].astype(str).str.lower().fillna("")
                    v["__untraceable"] = os_.str.contains("not_traceable|not traceable|untraceable", regex=True)
                else:
                    os_ = pd.Series("", index=v.index)
                    v["__untraceable"] = False
                rr = pd.to_numeric(v[risk], errors="coerce").fillna(0.25).clip(0, 1) if risk else pd.Series(0.25, index=v.index)
                w = (1.0 - rr) ** 2
                om = np.ones(len(v), float)
                if outcome:
                    om[os_.str.contains("met_borrower|met borrower", regex=True)] = 1.00
                    om[os_.str.contains("locked_premises|locked premises|locked", regex=True)] = 0.95
                    om[os_.str.contains("shifted|moved", regex=True)] = 0.55
                    om[os_.str.contains("not_traceable|not traceable|untraceable", regex=True)] = 0.05
                v["__w"] = np.maximum(0.01, np.asarray(w) * om)
                v["__clean"] = (~v["__hub"]) & (~v["__untraceable"])

                rows = []
                for aid, g in v.groupby(master_id):
                    cg = g[g["__clean"]]
                    if len(cg):
                        P = cg[[vx, vy]].to_numpy(float)
                        W = cg["__w"].to_numpy(float)
                        geom = weighted_geometric_median(P, W)
                        dist = np.linalg.norm(P - geom, axis=1)
                        spread = float(np.median(dist)) if len(dist) else np.nan
                        medrisk = float(pd.to_numeric(cg[risk], errors="coerce").median()) if risk else 0.25
                    else:
                        geom = np.array([np.nan, np.nan])
                        spread = medrisk = np.nan
                    rows.append({
                        master_id: str(aid),
                        "clean_visit_count": int(len(cg)),
                        "all_visit_count": int(len(g)),
                        "visit_spread_m": spread,
                        "median_integrity_risk": medrisk,
                        "visit_pin_x": geom[0],
                        "visit_pin_y": geom[1],
                    })
                va = pd.DataFrame(rows)
                out = addr.merge(va, how="left", on=master_id)
                out["clean_visit_count"] = pd.to_numeric(out["clean_visit_count"], errors="coerce").fillna(0).astype(int)
                out["all_visit_count"] = pd.to_numeric(out["all_visit_count"], errors="coerce").fillna(0).astype(int)
                return out, f"raw_visit_file:{OPTIONAL_VISIT_PATH}"
        except Exception as e:
            print(f"NOTE: raw visit file could not be used; falling back to pt_x/pt_y. Reason: {e}")

    # Fallback: Stage-1 pseudo-truth is already the clean visit consensus.
    out = addr.copy()
    cc = "clean_visits" if "clean_visits" in out.columns else ("pt_k" if "pt_k" in out.columns else None)
    tc = "total_visits" if "total_visits" in out.columns else cc
    if cc:
        out["clean_visit_count"] = pd.to_numeric(out[cc], errors="coerce").fillna(0).astype(int)
    else:
        out["clean_visit_count"] = out[[px, py]].notna().all(axis=1).astype(int)
    out["all_visit_count"] = pd.to_numeric(out[tc], errors="coerce").fillna(0).astype(int) if tc else out["clean_visit_count"]
    out["visit_spread_m"] = pd.to_numeric(out["pt_spread_m"], errors="coerce") if "pt_spread_m" in out.columns else np.nan
    out["median_integrity_risk"] = pd.to_numeric(out["avg_integrity_risk"], errors="coerce") if "avg_integrity_risk" in out.columns else np.nan
    out["visit_pin_x"] = pd.to_numeric(out[px], errors="coerce")
    out["visit_pin_y"] = pd.to_numeric(out[py], errors="coerce")
    return out, "address_level_pt_consensus"


def build_component_predictions(addr: pd.DataFrame, pseudo_train: pd.DataFrame,
                                px: str, py: str, bx: Optional[str], by: Optional[str],
                                poix: Optional[str], poiy: Optional[str], loc: Optional[str]):
    # Drop all-null features before pipelines; this removes lm_conf warning.
    grid_num = usable_num_cols(pseudo_train, STRUCTURED_NUM_CANDIDATES)
    grid_cat = usable_cat_cols(pseudo_train, STRUCTURED_CAT_CANDIDATES)

    full_num_candidates = STRUCTURED_NUM_CANDIDATES + EXTRA_NUM_CANDIDATES
    if bx: full_num_candidates.append(bx)
    if by: full_num_candidates.append(by)
    if poix: full_num_candidates.append(poix)
    if poiy: full_num_candidates.append(poiy)
    full_num = usable_num_cols(pseudo_train, full_num_candidates)
    full_cat = usable_cat_cols(pseudo_train, STRUCTURED_CAT_CANDIDATES)

    # 1) robust street-grid model
    grid_models = fit_pair(pseudo_train, grid_num, grid_cat, px, py, "grid_huber")
    grid_pred = pair_predict(grid_models, addr, grid_num + grid_cat)

    # 2) residual GBM around commercial geocoder
    resid_pred = np.full((len(addr), 2), np.nan)
    if bx and by and bx in pseudo_train.columns and by in pseudo_train.columns:
        tr = pseudo_train.loc[pseudo_train[[bx, by, px, py]].notna().all(axis=1)].copy()
        tr["__dx"] = tr[px] - tr[bx]
        tr["__dy"] = tr[py] - tr[by]
        rx, ry = fit_pair(tr, full_num, full_cat, "__dx", "__dy", "resid_hgb")
        if rx is not None:
            X = addr[full_num + full_cat]
            dx = rx.predict(X); dy = ry.predict(X)
            base_ok = addr[[bx, by]].notna().all(axis=1).to_numpy()
            resid_pred[base_ok, 0] = addr.loc[base_ok, bx].to_numpy(float) + dx[base_ok]
            resid_pred[base_ok, 1] = addr.loc[base_ok, by].to_numpy(float) + dy[base_ok]

    # 3) A3 direct ExtraTrees
    extra_models = fit_pair(pseudo_train, full_num, full_cat, px, py, "extra")
    extra_pred = pair_predict(extra_models, addr, full_num + full_cat)

    # 4) POI and 5) baseline
    poi_pred = np.full((len(addr), 2), np.nan)
    if poix and poiy:
        poi_pred[:, 0] = pd.to_numeric(addr[poix], errors="coerce")
        poi_pred[:, 1] = pd.to_numeric(addr[poiy], errors="coerce")

    baseline_pred = np.full((len(addr), 2), np.nan)
    if bx and by:
        baseline_pred[:, 0] = pd.to_numeric(addr[bx], errors="coerce")
        baseline_pred[:, 1] = pd.to_numeric(addr[by], errors="coerce")

    # fallback locality centroid from pseudo-truth training addresses
    locality_pred = np.full((len(addr), 2), np.nan)
    if loc and loc in addr.columns and loc in pseudo_train.columns:
        centers = pseudo_train.groupby(loc)[[px, py]].median()
        for i, val in enumerate(addr[loc]):
            if pd.notna(val) and val in centers.index:
                locality_pred[i] = centers.loc[val].to_numpy(float)
    global_center = pseudo_train[[px, py]].median().to_numpy(float)

    components = {
        "grid": grid_pred,
        "resid": resid_pred,
        "extra": extra_pred,
        "poi": poi_pred,
        "baseline": baseline_pred,
    }
    meta = {
        "grid_num": grid_num,
        "grid_cat": grid_cat,
        "full_num": full_num,
        "full_cat": full_cat,
    }
    return components, locality_pred, global_center, meta


def component_disagreement(component_points: Dict[str, np.ndarray], i: int) -> Tuple[float, int]:
    pts = []
    for arr in component_points.values():
        p = arr[i]
        if np.all(np.isfinite(p)):
            pts.append(p)
    if len(pts) < 2:
        return (250.0 if len(pts) == 1 else 600.0), len(pts)
    P = np.vstack(pts)
    d = []
    for a in range(len(P)):
        for b in range(a + 1, len(P)):
            d.append(float(np.linalg.norm(P[a] - P[b])))
    return float(np.median(d)), len(pts)


def inverse_error_weights(cal: pd.DataFrame, source_names: Sequence[str], tx: str, ty: str) -> Dict[str, float]:
    med = {}
    for name in source_names:
        xcol, ycol = f"cand_{name}_x", f"cand_{name}_y"
        m = cal[[xcol, ycol, tx, ty]].notna().all(axis=1)
        if m.sum() >= 3:
            err = euclid(cal.loc[m, xcol], cal.loc[m, ycol], cal.loc[m, tx], cal.loc[m, ty])
            med[name] = float(np.median(err))
    defaults = {"grid": 0.18, "resid": 0.18, "extra": 0.22, "poi": 0.30, "baseline": 0.12}
    if len(med) < 2:
        return defaults
    inv = {k: 1.0 / max(v, 20.0) for k, v in med.items()}
    w = {k: inv.get(k, 0.02) for k in defaults}
    z = sum(w.values())
    return {k: v / z for k, v in w.items()}


def weighted_component_ensemble(frame: pd.DataFrame, weights: Dict[str, float], source_names: Sequence[str]) -> np.ndarray:
    out = np.full((len(frame), 2), np.nan)
    for j, (_, row) in enumerate(frame.iterrows()):
        pts, ws = [], []
        for name in source_names:
            p = np.array([row.get(f"cand_{name}_x", np.nan), row.get(f"cand_{name}_y", np.nan)], float)
            if np.all(np.isfinite(p)) and weights.get(name, 0) > 0:
                pts.append(p); ws.append(weights[name])
        if pts:
            P = np.vstack(pts); W = np.asarray(ws, float); W /= W.sum()
            out[j] = np.sum(P * W[:, None], axis=0)
    return out


def fit_tail_gate(cal_no_visit: pd.DataFrame, base_xy: np.ndarray, tx: str, ty: str):
    """Tune how strongly high-disagreement A4 predictions are pulled toward A3 ExtraTrees."""
    if len(cal_no_visit) < 8:
        return {"alpha_max": 0.5, "q_lo": 0.50, "q_hi": 0.90, "d_lo": 200.0, "d_hi": 800.0}

    d = pd.to_numeric(cal_no_visit["component_disagreement_m"], errors="coerce").fillna(600).to_numpy(float)
    extra = cal_no_visit[["cand_extra_x", "cand_extra_y"]].to_numpy(float)
    truth = cal_no_visit[[tx, ty]].to_numpy(float)
    valid_extra = np.all(np.isfinite(extra), axis=1)

    best = None
    for q_lo in [0.40, 0.50, 0.60]:
        for q_hi in [0.80, 0.90, 0.95]:
            if q_hi <= q_lo:
                continue
            dlo, dhi = np.quantile(d, [q_lo, q_hi])
            if dhi <= dlo + 1e-9:
                dhi = dlo + 1.0
            risk = np.clip((d - dlo) / (dhi - dlo), 0, 1)
            for alpha_max in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:
                alpha = alpha_max * risk
                pred = base_xy.copy()
                ok = valid_extra & np.all(np.isfinite(pred), axis=1)
                pred[ok] = (1 - alpha[ok, None]) * pred[ok] + alpha[ok, None] * extra[ok]
                e = np.linalg.norm(pred - truth, axis=1)
                e = e[np.isfinite(e)]
                if len(e) < 5:
                    continue
                score = (TAIL_P90_WEIGHT * np.quantile(e, 0.90) +
                         TAIL_MEDIAN_WEIGHT * np.median(e) +
                         TAIL_MEAN_WEIGHT * np.mean(e))
                candidate = (score, alpha_max, q_lo, q_hi, float(dlo), float(dhi),
                             float(np.median(e)), float(np.quantile(e, 0.90)))
                if best is None or candidate[0] < best[0]:
                    best = candidate
    if best is None:
        return {"alpha_max": 0.5, "q_lo": 0.50, "q_hi": 0.90, "d_lo": 200.0, "d_hi": 800.0}
    return {
        "alpha_max": best[1], "q_lo": best[2], "q_hi": best[3],
        "d_lo": best[4], "d_hi": best[5],
        "cal_median_m": best[6], "cal_p90_m": best[7],
    }


def apply_tail_gate(frame: pd.DataFrame, base_xy: np.ndarray, gate: Dict[str, float]) -> Tuple[np.ndarray, np.ndarray]:
    d = pd.to_numeric(frame["component_disagreement_m"], errors="coerce").fillna(600).to_numpy(float)
    extra = frame[["cand_extra_x", "cand_extra_y"]].to_numpy(float)
    denom = max(gate["d_hi"] - gate["d_lo"], 1.0)
    risk = np.clip((d - gate["d_lo"]) / denom, 0, 1)
    alpha = gate["alpha_max"] * risk
    out = base_xy.copy()
    ok = np.all(np.isfinite(out), axis=1) & np.all(np.isfinite(extra), axis=1)
    out[ok] = (1 - alpha[ok, None]) * out[ok] + alpha[ok, None] * extra[ok]
    return out, alpha


def fallback_no_visit(frame: pd.DataFrame, xy: np.ndarray, locality_lookup: Dict[str, np.ndarray], global_center: np.ndarray,
                      loc: Optional[str], bx: Optional[str], by: Optional[str]) -> Tuple[np.ndarray, List[str]]:
    out = xy.copy()
    methods = []
    for j, (_, row) in enumerate(frame.iterrows()):
        if np.all(np.isfinite(out[j])):
            methods.append("a4_ensemble+a3_tail_gate")
            continue
        if bx and by and pd.notna(row.get(bx)) and pd.notna(row.get(by)):
            out[j] = [float(row[bx]), float(row[by])]
            methods.append("commercial_fallback")
        elif loc and pd.notna(row.get(loc)) and str(row.get(loc)) in locality_lookup:
            out[j] = locality_lookup[str(row.get(loc))]
            methods.append("locality_fallback")
        else:
            out[j] = global_center
            methods.append("global_centroid_fallback")
    return out, methods


def confidence_feature_lists(df: pd.DataFrame, loc: Optional[str]):
    nums = usable_num_cols(df, [
        "clean_visit_count", "all_visit_count", "visit_spread_m", "median_integrity_risk",
        "component_disagreement_m", "component_source_count", "tail_gate_alpha",
        "locality_conf", "n_fields_found", "script_kn_frac", "script_dev_frac",
        "is_village_style", "dist_pred_to_baseline_m", "dist_pred_to_poi_m"
    ])
    cats = usable_cat_cols(df, ["evidence_tier", loc] if loc else ["evidence_tier"])
    return nums, cats


def fit_confidence_model(train: pd.DataFrame, err_col: str, loc: Optional[str]):
    """A3-style 90th-quantile model with a held-out conformal correction.

    The quantile regressor learns conditional error size. A separate calibration
    slice estimates an additive correction, which is much more reliable than
    calibrating on the same rows used to fit the quantile model.
    """
    num, cat = confidence_feature_lists(train, loc)
    features = num + cat
    tr = train.loc[train[err_col].notna()].copy()
    if len(tr) < 30 or not features:
        return None, num, cat, conformal_quantile(tr[err_col])

    indices = np.arange(len(tr))
    strat = (tr["clean_visit_count"].to_numpy() > 0).astype(int) if "clean_visit_count" in tr.columns else None
    try:
        fit_idx, cal_idx = train_test_split(
            indices, test_size=0.25, random_state=SEED, stratify=strat
        )
    except ValueError:
        fit_idx, cal_idx = train_test_split(indices, test_size=0.25, random_state=SEED)

    qfit = tr.iloc[fit_idx]
    qcal = tr.iloc[cal_idx]
    model = make_pipeline("quantile", num, cat)
    model.fit(qfit[features], qfit[err_col])

    raw_cal = np.maximum(model.predict(qcal[features]), 10.0)
    nonconformity = np.maximum(qcal[err_col].to_numpy(float) - raw_cal, 0.0)
    additive = conformal_quantile(nonconformity)
    if not np.isfinite(additive):
        additive = 0.0
    return model, num, cat, float(additive)


def predict_radius(model_pack, df: pd.DataFrame) -> np.ndarray:
    model, num, cat, correction = model_pack
    if model is None:
        return np.full(len(df), max(float(correction) if np.isfinite(correction) else 500.0, 10.0))
    raw = np.maximum(model.predict(df[num + cat]), 10.0)
    return np.maximum(raw + correction, 10.0)


OUTPUT_DIR = "stage2_arch51"
ROUTER_FOLDS = 5
ROUTER_SOURCES = ["grid", "resid", "extra", "baseline"]
ROUTER_MIN_ROWS = 80


ROUTER_P90_WEIGHT = 1.0
ROUTER_MEDIAN_WEIGHT = 0.25
ROUTER_MEAN_WEIGHT = 0.05
ROUTER_CATASTROPHIC_WEIGHT = 120.0  


def add_candidate_columns(frame: pd.DataFrame, components: Dict[str, np.ndarray]) -> pd.DataFrame:
    out = frame.copy()
    for name, arr in components.items():
        out[f"cand_{name}_x"] = arr[:, 0]
        out[f"cand_{name}_y"] = arr[:, 1]
    return out


def build_support_features(reference: pd.DataFrame, target: pd.DataFrame,
                           grid_num: Sequence[str], loc: Optional[str]) -> pd.DataFrame:
    """Quantify how well each target address is supported by pseudo-truth training data.

    The features are deliberately address-only: no true coordinates are used.
    They measure locality density, street-field coverage, extrapolation beyond the
    known locality range, and nearest distance in normalized street-number space.
    """
    out = pd.DataFrame(index=target.index)
    grid_num = [c for c in grid_num if c in target.columns and c in reference.columns]

    out["router_grid_fields_count"] = 0.0
    for c in grid_num:
        out["router_grid_fields_count"] += pd.to_numeric(target[c], errors="coerce").notna().astype(float)

    if loc and loc in reference.columns and loc in target.columns:
        counts = reference.groupby(loc).size().to_dict()
        out["router_locality_train_count"] = target[loc].map(counts).fillna(0).astype(float)
    else:
        out["router_locality_train_count"] = float(len(reference))

    extrap_count = []
    extrap_score = []
    nearest_dist = []

    if loc and loc in reference.columns:
        groups = {str(k): g for k, g in reference.groupby(loc, dropna=False)}
    else:
        groups = {"__all__": reference}

    for _, row in target.iterrows():
        if loc and loc in target.columns and pd.notna(row.get(loc)):
            ref = groups.get(str(row.get(loc)), reference.iloc[0:0])
        else:
            ref = reference
        if len(ref) == 0:
            ref = reference

        present = []
        scales = {}
        medians = {}
        ex_count = 0
        ex_score = 0.0

        for c in grid_num:
            tv = pd.to_numeric(pd.Series([row.get(c)]), errors="coerce").iloc[0]
            if not np.isfinite(tv):
                continue
            rv = pd.to_numeric(ref[c], errors="coerce").dropna().to_numpy(float)
            if len(rv) < 3:
                continue
            q25, q75 = np.quantile(rv, [0.25, 0.75])
            scale = max(float(q75 - q25), float(np.std(rv)), 1.0)
            lo, hi = float(np.min(rv)), float(np.max(rv))
            if tv < lo:
                ex_count += 1
                ex_score += (lo - tv) / scale
            elif tv > hi:
                ex_count += 1
                ex_score += (tv - hi) / scale
            present.append(c)
            scales[c] = scale
            medians[c] = float(np.median(rv))

        extrap_count.append(float(ex_count))
        extrap_score.append(float(ex_score))

        if not present or len(ref) == 0:
            nearest_dist.append(np.nan)
            continue

        
        cols = present
        R = np.zeros((len(ref), len(cols)), float)
        T = np.zeros(len(cols), float)
        for j, c in enumerate(cols):
            vals = pd.to_numeric(ref[c], errors="coerce").to_numpy(float)
            vals = np.where(np.isfinite(vals), vals, medians[c])
            R[:, j] = (vals - medians[c]) / scales[c]
            tv = float(pd.to_numeric(pd.Series([row.get(c)]), errors="coerce").iloc[0])
            T[j] = (tv - medians[c]) / scales[c]
        d = np.sqrt(np.mean((R - T[None, :]) ** 2, axis=1))
        nearest_dist.append(float(np.min(d)) if len(d) else np.nan)

    out["router_grid_extrapolation_count"] = extrap_count
    out["router_grid_extrapolation_score"] = extrap_score
    out["router_nearest_grid_feature_distance"] = nearest_dist
    return out


def add_router_meta_features(frame: pd.DataFrame, bx: Optional[str], by: Optional[str]) -> pd.DataFrame:
    out = frame.copy()
    pairs = [
        ("grid", "extra"), ("grid", "resid"), ("grid", "baseline"),
        ("extra", "resid"), ("extra", "baseline"), ("resid", "baseline"),
    ]
    dcols = []
    for a, b in pairs:
        c = f"router_d_{a}_{b}_m"
        ax, ay = f"cand_{a}_x", f"cand_{a}_y"
        bxcol, bycol = f"cand_{b}_x", f"cand_{b}_y"
        if all(x in out.columns for x in [ax, ay, bxcol, bycol]):
            ok = out[[ax, ay, bxcol, bycol]].notna().all(axis=1)
            out[c] = np.nan
            if ok.any():
                out.loc[ok, c] = euclid(out.loc[ok, ax], out.loc[ok, ay], out.loc[ok, bxcol], out.loc[ok, bycol])
        else:
            out[c] = np.nan
        dcols.append(c)

    out["router_component_disagreement_m"] = out[dcols].median(axis=1, skipna=True)
    out["router_source_count"] = 0.0
    for s in ROUTER_SOURCES:
        ok = out[[f"cand_{s}_x", f"cand_{s}_y"]].notna().all(axis=1) if f"cand_{s}_x" in out.columns else pd.Series(False, index=out.index)
        out[f"router_has_{s}"] = ok.astype(float)
        out["router_source_count"] += ok.astype(float)

    if bx and by and bx in out.columns and by in out.columns:
        # Raw commercial coordinates help the risk model learn location-specific regimes.
        out["router_baseline_x"] = pd.to_numeric(out[bx], errors="coerce")
        out["router_baseline_y"] = pd.to_numeric(out[by], errors="coerce")
    return out


def router_feature_lists(df: pd.DataFrame, loc: Optional[str]) -> Tuple[List[str], List[str]]:
    num_candidates = [
        "router_grid_fields_count", "router_locality_train_count",
        "router_grid_extrapolation_count", "router_grid_extrapolation_score",
        "router_nearest_grid_feature_distance", "router_component_disagreement_m",
        "router_source_count", "router_d_grid_extra_m", "router_d_grid_resid_m",
        "router_d_grid_baseline_m", "router_d_extra_resid_m",
        "router_d_extra_baseline_m", "router_d_resid_baseline_m",
        "router_has_grid", "router_has_resid", "router_has_extra", "router_has_baseline",
        "router_baseline_x", "router_baseline_y", "locality_conf", "n_fields_found",
        "script_kn_frac", "script_dev_frac", "is_village_style",
    ] + STRUCTURED_NUM_CANDIDATES
    cat_candidates = [
        loc, "locality_method", "town_id", "town_method", "base_precision", "block", "rel_no_type", "rel_context"
    ] if loc else ["locality_method", "town_id", "town_method", "base_precision", "block", "rel_no_type", "rel_context"]
    return usable_num_cols(df, num_candidates), usable_cat_cols(df, [c for c in cat_candidates if c])


def build_pseudo_oof_router_data(addr: pd.DataFrame, pseudo_train: pd.DataFrame,
                                 px: str, py: str, bx: Optional[str], by: Optional[str],
                                 poix: Optional[str], poiy: Optional[str], loc: Optional[str]) -> pd.DataFrame:
    """Create honest simulated-no-visit training examples for the safety controller.

    Every pseudo-truth address is predicted out-of-fold after its visit truth is hidden.
    POI coordinates are blanked on the target side so the router learns the exact NV_grid
    regime we are trying to fix rather than learning from easier POI-anchored cases.
    """
    kf = KFold(n_splits=ROUTER_FOLDS, shuffle=True, random_state=SEED)
    chunks = []
    p = pseudo_train.reset_index(drop=True).copy()

    for fold, (tr_idx, va_idx) in enumerate(kf.split(p), 1):
        ref = p.iloc[tr_idx].copy()
        val_true = p.iloc[va_idx].copy()
        val_sim = val_true.copy()
        if poix and poix in val_sim.columns:
            val_sim[poix] = np.nan
        if poiy and poiy in val_sim.columns:
            val_sim[poiy] = np.nan
        if "poi_id" in val_sim.columns:
            val_sim["poi_id"] = np.nan

        comps, _, _, meta = build_component_predictions(val_sim, ref, px, py, bx, by, poix, poiy, loc)
        f = add_candidate_columns(val_sim, comps)
        supp = build_support_features(ref, val_sim, meta["grid_num"], loc)
        for c in supp.columns:
            f[c] = supp[c].to_numpy()
        f = add_router_meta_features(f, bx, by)
        f["router_fold"] = fold
        f["router_truth_x"] = val_true[px].to_numpy(float)
        f["router_truth_y"] = val_true[py].to_numpy(float)

        for s in ROUTER_SOURCES:
            ok = f[[f"cand_{s}_x", f"cand_{s}_y", "router_truth_x", "router_truth_y"]].notna().all(axis=1)
            f[f"router_error_{s}_m"] = np.nan
            if ok.any():
                f.loc[ok, f"router_error_{s}_m"] = euclid(
                    f.loc[ok, f"cand_{s}_x"], f.loc[ok, f"cand_{s}_y"],
                    f.loc[ok, "router_truth_x"], f.loc[ok, "router_truth_y"]
                )
        # Only examples that actually have some grid/street information are relevant.
        f = f.loc[f["router_grid_fields_count"] > 0].copy()
        chunks.append(f)

    if not chunks:
        raise RuntimeError("Could not construct pseudo-truth OOF router training data.")
    return pd.concat(chunks, ignore_index=True)


def fit_risk_models(train: pd.DataFrame, sources: Sequence[str], num: List[str], cat: List[str]):
    features = num + cat
    models = {}
    medians = {}
    for s in sources:
        ycol = f"router_error_{s}_m"
        tr = train.loc[train[ycol].notna()].copy()
        if len(tr) < 30:
            models[s] = None
            medians[s] = 600.0
            continue
        y = np.log1p(np.maximum(tr[ycol].to_numpy(float), 0.0))
        model = make_pipeline("extra", num, cat)
        model.fit(tr[features], y)
        models[s] = model
        medians[s] = float(np.median(tr[ycol]))
    inv = {s: 1.0 / max(medians.get(s, 600.0), 25.0) for s in sources}
    z = sum(inv.values()) or 1.0
    priors = {s: inv[s] / z for s in sources}
    return models, priors


def predict_risks(models: Dict[str, object], frame: pd.DataFrame,
                  sources: Sequence[str], features: List[str]) -> Dict[str, np.ndarray]:
    out = {}
    for s in sources:
        m = models.get(s)
        if m is None:
            out[s] = np.full(len(frame), 800.0)
        else:
            pred = np.expm1(m.predict(frame[features]))
            out[s] = np.clip(pred, 5.0, 5000.0)
    return out


def router_blend(frame: pd.DataFrame, risk_pred: Dict[str, np.ndarray], priors: Dict[str, float],
                 power: float, baseline_floor: float, extrap_penalty: float):
    n = len(frame)
    pred = np.full((n, 2), np.nan)
    weights = {s: np.zeros(n, float) for s in ROUTER_SOURCES}

    extrap = pd.to_numeric(frame.get("router_grid_extrapolation_score", 0.0), errors="coerce").fillna(0).to_numpy(float)

    for i, (_, row) in enumerate(frame.iterrows()):
        raw = {}
        pts = {}
        for s in ROUTER_SOURCES:
            p = np.array([row.get(f"cand_{s}_x", np.nan), row.get(f"cand_{s}_y", np.nan)], float)
            if not np.all(np.isfinite(p)):
                continue
            pts[s] = p
            risk = max(float(risk_pred[s][i]), 10.0)
            score = float(priors.get(s, 0.01)) / (risk ** power)
            if s == "grid":
                score *= math.exp(-extrap_penalty * min(float(extrap[i]), 8.0))
            raw[s] = max(score, 1e-12)
        if not raw:
            continue
        z = sum(raw.values())
        w = {s: raw[s] / z for s in raw}

        # A small commercial-baseline floor protects unsupported extrapolation cases.
        if "baseline" in w and baseline_floor > 0 and w["baseline"] < baseline_floor:
            other = 1.0 - w["baseline"]
            if other > 1e-12:
                scale = (1.0 - baseline_floor) / other
                for s in list(w):
                    if s != "baseline":
                        w[s] *= scale
                w["baseline"] = baseline_floor

        pp = np.zeros(2, float)
        for s, ww in w.items():
            pp += ww * pts[s]
            weights[s][i] = ww
        pred[i] = pp
    return pred, weights


def router_objective(errors: np.ndarray) -> float:
    e = np.asarray(errors, float)
    e = e[np.isfinite(e)]
    if len(e) == 0:
        return np.inf
    catastrophic_rate = float(np.mean(e > 1000.0))
    return (ROUTER_P90_WEIGHT * float(np.quantile(e, 0.90)) +
            ROUTER_MEDIAN_WEIGHT * float(np.median(e)) +
            ROUTER_MEAN_WEIGHT * float(np.mean(e)) +
            ROUTER_CATASTROPHIC_WEIGHT * catastrophic_rate)


def tune_and_fit_router(router_oof: pd.DataFrame, loc: Optional[str]):
    num, cat = router_feature_lists(router_oof, loc)
    features = num + cat
    if len(router_oof) < ROUTER_MIN_ROWS or not features:
        raise RuntimeError(f"Insufficient router training data: {len(router_oof)} rows, {len(features)} features")

    idx = np.arange(len(router_oof))
    fit_idx, val_idx = train_test_split(idx, test_size=0.25, random_state=SEED)
    fit = router_oof.iloc[fit_idx].copy()
    val = router_oof.iloc[val_idx].copy()

    models, priors = fit_risk_models(fit, ROUTER_SOURCES, num, cat)
    risks = predict_risks(models, val, ROUTER_SOURCES, features)
    truth = val[["router_truth_x", "router_truth_y"]].to_numpy(float)

    best = None
    trials = []
    for power in [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]:
        for floor in [0.0, 0.05, 0.10, 0.15, 0.20, 0.30]:
            for penalty in [0.0, 0.25, 0.5, 1.0, 1.5, 2.0]:
                p, _ = router_blend(val, risks, priors, power, floor, penalty)
                e = np.linalg.norm(p - truth, axis=1)
                score = router_objective(e)
                row = {
                    "score": score, "power": power, "baseline_floor": floor,
                    "extrap_penalty": penalty, "median_m": float(np.nanmedian(e)),
                    "p90_m": float(np.nanquantile(e, 0.90)), "mean_m": float(np.nanmean(e)),
                    "catastrophic_rate": float(np.nanmean(e > 1000.0)),
                }
                trials.append(row)
                if best is None or score < best["score"]:
                    best = row

    # Refit risk models on all honest pseudo-OOF examples after tuning.
    final_models, final_priors = fit_risk_models(router_oof, ROUTER_SOURCES, num, cat)
    return {
        "models": final_models,
        "priors": final_priors,
        "num": num,
        "cat": cat,
        "features": features,
        "params": {
            "power": float(best["power"]),
            "baseline_floor": float(best["baseline_floor"]),
            "extrap_penalty": float(best["extrap_penalty"]),
        },
        "tuning_best": best,
        "tuning_trials": pd.DataFrame(trials).sort_values("score").reset_index(drop=True),
    }


def apply_router_pack(pack: Dict, frame: pd.DataFrame):
    risks = predict_risks(pack["models"], frame, ROUTER_SOURCES, pack["features"])
    p, weights = router_blend(
        frame, risks, pack["priors"],
        pack["params"]["power"], pack["params"]["baseline_floor"], pack["params"]["extrap_penalty"]
    )
    return p, weights, risks



A52_OUTPUT_DIR = "stage2_arch52"
A52_ROUTER_CV_FOLDS = 3


A52_P90_WEIGHT = 1.0
A52_CVAR90_WEIGHT = 0.35
A52_MEDIAN_WEIGHT = 0.10
A52_MEAN_WEIGHT = 0.02
A52_CATASTROPHIC_WEIGHT = 500.0


def a52_cvar(errors: np.ndarray, q: float = 0.90) -> float:
    e = np.asarray(errors, float)
    e = e[np.isfinite(e)]
    if len(e) == 0:
        return np.inf
    cut = float(np.quantile(e, q))
    tail = e[e >= cut]
    return float(np.mean(tail)) if len(tail) else cut


def a52_router_objective(errors: np.ndarray) -> float:
    e = np.asarray(errors, float)
    e = e[np.isfinite(e)]
    if len(e) == 0:
        return np.inf
    p90 = float(np.quantile(e, 0.90))
    med = float(np.median(e))
    mean = float(np.mean(e))
    cvar = a52_cvar(e, 0.90)
    catastrophic_rate = float(np.mean(e > 1000.0))
    return (A52_P90_WEIGHT * p90 +
            A52_CVAR90_WEIGHT * cvar +
            A52_MEDIAN_WEIGHT * med +
            A52_MEAN_WEIGHT * mean +
            A52_CATASTROPHIC_WEIGHT * catastrophic_rate)


def a52_risk_cv_predictions(router_oof: pd.DataFrame, loc: Optional[str]):
    """Create honest CV predictions of each expert's error risk.

    Hyperparameters for expert routing are tuned only against these OOF risk
    predictions, never against the surveyed evaluation rows.
    """
    num, cat = router_feature_lists(router_oof, loc)
    features = num + cat
    if len(router_oof) < ROUTER_MIN_ROWS or not features:
        raise RuntimeError(
            f"Insufficient router training data: {len(router_oof)} rows, {len(features)} features"
        )

    kf = KFold(n_splits=A52_ROUTER_CV_FOLDS, shuffle=True, random_state=SEED + 17)
    n = len(router_oof)
    risk_oof = {s: np.full(n, np.nan) for s in ROUTER_SOURCES}

    for tr_idx, va_idx in kf.split(np.arange(n)):
        tr = router_oof.iloc[tr_idx].copy()
        va = router_oof.iloc[va_idx].copy()
        models, _ = fit_risk_models(tr, ROUTER_SOURCES, num, cat)
        rr = predict_risks(models, va, ROUTER_SOURCES, features)
        for s in ROUTER_SOURCES:
            risk_oof[s][va_idx] = rr[s]

    # Global priors are computed only from the pseudo-OOF expert errors.
    medians = {}
    for s in ROUTER_SOURCES:
        vals = pd.to_numeric(router_oof[f"router_error_{s}_m"], errors="coerce").dropna().to_numpy(float)
        medians[s] = float(np.median(vals)) if len(vals) else 600.0
    inv = {s: 1.0 / max(medians[s], 25.0) for s in ROUTER_SOURCES}
    z = sum(inv.values()) or 1.0
    priors = {s: inv[s] / z for s in ROUTER_SOURCES}
    return risk_oof, priors, num, cat, features


def a52_nearest_threshold(frame: pd.DataFrame, q: float) -> float:
    if q > 1.0:
        return np.inf
    vals = pd.to_numeric(
        frame.get("router_nearest_grid_feature_distance", pd.Series(dtype=float)),
        errors="coerce",
    ).dropna().to_numpy(float)
    if len(vals) == 0:
        return np.inf
    return float(np.quantile(vals, q))


def a52_select_route(
    frame: pd.DataFrame,
    risk_pred: Dict[str, np.ndarray],
    priors: Dict[str, float],
    params: Dict[str, object],
):
    """Top-1/top-2 expert router with an explicit OOD mode.

    Unlike Architecture 5.1, this function never blends all four experts.
    It either selects the predicted safest expert or blends only the two safest.
    """
    n = len(frame)
    pred = np.full((n, 2), np.nan)
    weights = {s: np.zeros(n, float) for s in ROUTER_SOURCES}
    adjusted = {s: np.full(n, np.nan) for s in ROUTER_SOURCES}
    chosen1 = np.array([""] * n, dtype=object)
    chosen2 = np.array([""] * n, dtype=object)
    ood_flag = np.zeros(n, dtype=bool)
    ood_reason = np.array([""] * n, dtype=object)

    count = pd.to_numeric(frame.get("router_locality_train_count", 0.0), errors="coerce").fillna(0).to_numpy(float)
    nearest = pd.to_numeric(frame.get("router_nearest_grid_feature_distance", np.nan), errors="coerce").to_numpy(float)
    extrap = pd.to_numeric(frame.get("router_grid_extrapolation_score", 0.0), errors="coerce").fillna(0).to_numpy(float)

    prior_strength = float(params["prior_strength"])
    top1_ratio = float(params["top1_ratio"])
    top2_power = float(params["top2_power"])
    support_penalty = float(params["support_penalty"])
    count_thr = float(params["ood_count_threshold"])
    nearest_thr = float(params["ood_nearest_threshold"])
    extrap_thr = float(params["ood_extrap_threshold"])
    ood_policy = str(params["ood_policy"])

    for i, (_, row) in enumerate(frame.iterrows()):
        pts = {}
        arisk = {}
        for s in ROUTER_SOURCES:
            p = np.array([row.get(f"cand_{s}_x", np.nan), row.get(f"cand_{s}_y", np.nan)], float)
            if not np.all(np.isfinite(p)):
                continue
            pts[s] = p
            r = max(float(risk_pred[s][i]), 5.0)
            prior = max(float(priors.get(s, 0.25)), 1e-4)
            r_adj = r / (prior ** prior_strength)


            low_support_term = 1.0 / math.sqrt(max(count[i], 0.0) + 1.0)
            if s == "grid":
                r_adj *= 1.0 + support_penalty * (min(extrap[i], 8.0) + 2.0 * low_support_term)
            elif s == "resid":
                r_adj *= 1.0 + 0.5 * support_penalty * (min(extrap[i], 8.0) + low_support_term)
            arisk[s] = max(float(r_adj), 5.0)
            adjusted[s][i] = arisk[s]

        if not pts:
            continue

        reasons = []
        if count[i] <= count_thr:
            reasons.append("low_locality_support")
        if np.isfinite(nearest[i]) and nearest[i] >= nearest_thr:
            reasons.append("far_grid_support")
        if extrap[i] >= extrap_thr:
            reasons.append("grid_extrapolation")
        is_ood = len(reasons) > 0
        ood_flag[i] = is_ood
        ood_reason[i] = "+".join(reasons)

        allowed = list(pts.keys())
        if is_ood:
            if ood_policy == "extra_baseline":
                preferred = [s for s in ["extra", "baseline"] if s in pts]
                if preferred:
                    allowed = preferred
            elif ood_policy == "no_grid":
                preferred = [s for s in ["extra", "resid", "baseline"] if s in pts]
                if preferred:
                    allowed = preferred
            elif ood_policy == "extra_only":
                if "extra" in pts:
                    allowed = ["extra"]
                elif "baseline" in pts:
                    allowed = ["baseline"]
            elif ood_policy == "baseline_only":
                if "baseline" in pts:
                    allowed = ["baseline"]
            

        ranked = sorted(allowed, key=lambda s: arisk.get(s, np.inf))
        if not ranked:
            continue
        best = ranked[0]
        chosen1[i] = best

        if len(ranked) == 1:
            weights[best][i] = 1.0
            pred[i] = pts[best]
            continue

        second = ranked[1]
        chosen2[i] = second
        ratio = arisk[second] / max(arisk[best], 1e-9)
        if ratio >= top1_ratio:
            # Strong confidence gap: use the safest predicted expert directly.
            weights[best][i] = 1.0
            pred[i] = pts[best]
        else:
            # Ambiguous case: blend only top two, never all experts.
            sb = 1.0 / (max(arisk[best], 5.0) ** top2_power)
            ss = 1.0 / (max(arisk[second], 5.0) ** top2_power)
            z = sb + ss
            wb = sb / z
            ws = ss / z
            weights[best][i] = wb
            weights[second][i] = ws
            pred[i] = wb * pts[best] + ws * pts[second]

    return pred, weights, adjusted, chosen1, chosen2, ood_flag, ood_reason


def tune_and_fit_router_a52(router_oof: pd.DataFrame, loc: Optional[str]):
    risk_oof, priors, num, cat, features = a52_risk_cv_predictions(router_oof, loc)
    truth = router_oof[["router_truth_x", "router_truth_y"]].to_numpy(float)

    # Candidate OOD distance thresholds are derived only from pseudo-OOF support.
    nearest_qs = [0.90, 0.97, 1.01]  # 1.01 disables nearest-distance OOD detection.
    nearest_thrs = {q: a52_nearest_threshold(router_oof, q) for q in nearest_qs}

    trials = []
    best = None
    for prior_strength in [0.0, 0.5]:
        for top1_ratio in [1.25, 1.60, 2.00]:
            for top2_power in [1.0, 2.0]:
                for support_penalty in [0.0, 0.5]:
                    for count_thr in [0.0, 2.0]:
                        for q in nearest_qs:
                            for extrap_thr in [1.0, np.inf]:
                                for policy in ["all", "extra_baseline", "no_grid", "extra_only"]:
                                    params = {
                                        "prior_strength": prior_strength,
                                        "top1_ratio": top1_ratio,
                                        "top2_power": top2_power,
                                        "support_penalty": support_penalty,
                                        "ood_count_threshold": count_thr,
                                        "ood_nearest_quantile": q,
                                        "ood_nearest_threshold": nearest_thrs[q],
                                        "ood_extrap_threshold": extrap_thr,
                                        "ood_policy": policy,
                                    }
                                    p, _, _, _, _, ood, _ = a52_select_route(
                                        router_oof, risk_oof, priors, params
                                    )
                                    e = np.linalg.norm(p - truth, axis=1)
                                    score = a52_router_objective(e)
                                    row = {
                                        "score": float(score),
                                        "median_m": float(np.nanmedian(e)),
                                        "p90_m": float(np.nanquantile(e, 0.90)),
                                        "cvar90_m": float(a52_cvar(e)),
                                        "mean_m": float(np.nanmean(e)),
                                        "catastrophic_rate": float(np.nanmean(e > 1000.0)),
                                        "ood_rate": float(np.mean(ood)),
                                        **params,
                                    }
                                    
                                    if not np.isfinite(row["ood_extrap_threshold"]):
                                        row["ood_extrap_threshold"] = 1e12
                                    trials.append(row)
                                    if best is None or score < best["score"]:
                                        best = row.copy()

    if best is None:
        raise RuntimeError("Architecture 5.2 router tuning failed.")


    best_params = {
        k: best[k] for k in [
            "prior_strength", "top1_ratio", "top2_power", "support_penalty",
            "ood_count_threshold", "ood_nearest_quantile", "ood_nearest_threshold",
            "ood_extrap_threshold", "ood_policy",
        ]
    }
    if float(best_params["ood_extrap_threshold"]) >= 1e11:
        best_params["ood_extrap_threshold"] = np.inf


    final_models, final_priors = fit_risk_models(router_oof, ROUTER_SOURCES, num, cat)
    return {
        "models": final_models,
        "priors": final_priors,
        "num": num,
        "cat": cat,
        "features": features,
        "params": best_params,
        "tuning_best": best,
        "tuning_trials": pd.DataFrame(trials).sort_values("score").reset_index(drop=True),
    }


def apply_router_pack_a52(pack: Dict, frame: pd.DataFrame):
    risks = predict_risks(pack["models"], frame, ROUTER_SOURCES, pack["features"])
    p, weights, adjusted, top1, top2, ood, reason = a52_select_route(
        frame, risks, pack["priors"], pack["params"]
    )
    return p, weights, risks, adjusted, top1, top2, ood, reason


def make_a52_risk_pipeline(num_cols: List[str], cat_cols: List[str]):
    pre = build_preprocessor(num_cols, cat_cols, scale_numeric=False)
    model = ExtraTreesRegressor(
        n_estimators=240,
        min_samples_leaf=4,
        max_features=0.75,
        random_state=SEED + 91,
        n_jobs=-1,
    )
    return Pipeline([("pre", pre), ("model", model)])


def fit_risk_models_a52(train: pd.DataFrame, sources: Sequence[str], num: List[str], cat: List[str]):
    features = num + cat
    models = {}
    medians = {}
    for s in sources:
        ycol = f"router_error_{s}_m"
        tr = train.loc[train[ycol].notna()].copy()
        if len(tr) < 30:
            models[s] = None
            medians[s] = 600.0
            continue
        y = np.log1p(np.maximum(tr[ycol].to_numpy(float), 0.0))
        model = make_a52_risk_pipeline(num, cat)
        model.fit(tr[features], y)
        models[s] = model
        medians[s] = float(np.median(tr[ycol]))
    inv = {s: 1.0 / max(medians.get(s, 600.0), 25.0) for s in sources}
    z = sum(inv.values()) or 1.0
    priors = {s: inv[s] / z for s in sources}
    return models, priors


def a52_risk_cv_predictions(router_oof: pd.DataFrame, loc: Optional[str]):
    num, cat = router_feature_lists(router_oof, loc)
    features = num + cat
    if len(router_oof) < ROUTER_MIN_ROWS or not features:
        raise RuntimeError(
            f"Insufficient router training data: {len(router_oof)} rows, {len(features)} features"
        )

    kf = KFold(n_splits=A52_ROUTER_CV_FOLDS, shuffle=True, random_state=SEED + 17)
    n = len(router_oof)
    risk_oof = {s: np.full(n, np.nan) for s in ROUTER_SOURCES}

    for tr_idx, va_idx in kf.split(np.arange(n)):
        tr = router_oof.iloc[tr_idx].copy()
        va = router_oof.iloc[va_idx].copy()
        models, _ = fit_risk_models_a52(tr, ROUTER_SOURCES, num, cat)
        rr = predict_risks(models, va, ROUTER_SOURCES, features)
        for s in ROUTER_SOURCES:
            risk_oof[s][va_idx] = rr[s]

    medians = {}
    for s in ROUTER_SOURCES:
        vals = pd.to_numeric(router_oof[f"router_error_{s}_m"], errors="coerce").dropna().to_numpy(float)
        medians[s] = float(np.median(vals)) if len(vals) else 600.0
    inv = {s: 1.0 / max(medians[s], 25.0) for s in ROUTER_SOURCES}
    z = sum(inv.values()) or 1.0
    priors = {s: inv[s] / z for s in ROUTER_SOURCES}
    return risk_oof, priors, num, cat, features


def tune_and_fit_router_a52(router_oof: pd.DataFrame, loc: Optional[str]):
    risk_oof, priors, num, cat, features = a52_risk_cv_predictions(router_oof, loc)
    truth = router_oof[["router_truth_x", "router_truth_y"]].to_numpy(float)

    nearest_qs = [0.93, 1.01]  # 1.01 disables nearest-distance OOD detection.
    nearest_thrs = {q: a52_nearest_threshold(router_oof, q) for q in nearest_qs}

    trials = []
    best = None
    for prior_strength in [0.0, 0.5]:
        for top1_ratio in [1.40, 1.85]:
            for top2_power in [1.0, 2.0]:
                for support_penalty in [0.0, 0.5]:
                    for count_thr in [0.0, 2.0]:
                        for q in nearest_qs:
                            for extrap_thr in [1.0, np.inf]:
                                for policy in ["all", "extra_baseline", "no_grid"]:
                                    params = {
                                        "prior_strength": prior_strength,
                                        "top1_ratio": top1_ratio,
                                        "top2_power": top2_power,
                                        "support_penalty": support_penalty,
                                        "ood_count_threshold": count_thr,
                                        "ood_nearest_quantile": q,
                                        "ood_nearest_threshold": nearest_thrs[q],
                                        "ood_extrap_threshold": extrap_thr,
                                        "ood_policy": policy,
                                    }
                                    p, _, _, _, _, ood, _ = a52_select_route(
                                        router_oof, risk_oof, priors, params
                                    )
                                    e = np.linalg.norm(p - truth, axis=1)
                                    score = a52_router_objective(e)
                                    row = {
                                        "score": float(score),
                                        "median_m": float(np.nanmedian(e)),
                                        "p90_m": float(np.nanquantile(e, 0.90)),
                                        "cvar90_m": float(a52_cvar(e)),
                                        "mean_m": float(np.nanmean(e)),
                                        "catastrophic_rate": float(np.nanmean(e > 1000.0)),
                                        "ood_rate": float(np.mean(ood)),
                                        **params,
                                    }
                                    if not np.isfinite(row["ood_extrap_threshold"]):
                                        row["ood_extrap_threshold"] = 1e12
                                    trials.append(row)
                                    if best is None or score < best["score"]:
                                        best = row.copy()

    if best is None:
        raise RuntimeError("Architecture 5.2 router tuning failed.")

    best_params = {
        k: best[k] for k in [
            "prior_strength", "top1_ratio", "top2_power", "support_penalty",
            "ood_count_threshold", "ood_nearest_quantile", "ood_nearest_threshold",
            "ood_extrap_threshold", "ood_policy",
        ]
    }
    if float(best_params["ood_extrap_threshold"]) >= 1e11:
        best_params["ood_extrap_threshold"] = np.inf

    final_models, final_priors = fit_risk_models_a52(router_oof, ROUTER_SOURCES, num, cat)
    return {
        "models": final_models,
        "priors": final_priors,
        "num": num,
        "cat": cat,
        "features": features,
        "params": best_params,
        "tuning_best": best,
        "tuning_trials": pd.DataFrame(trials).sort_values("score").reset_index(drop=True),
    }


def main():
    outdir = Path(A52_OUTPUT_DIR)
    outdir.mkdir(parents=True, exist_ok=True)

    master = normalize_df_columns(pd.read_csv(MASTER_PATH))
    survey = normalize_df_columns(pd.read_csv(SURVEY_PATH))

    mid = require_col(find_col(master, "address_id", "master_address_id"), "master address id", master)
    sid = require_col(find_col(survey, "address_id", "survey_address_id"), "survey address id", survey)
    tx = require_col(find_col(survey, "truth_x", "truth_x"), "survey truth x", survey)
    ty = require_col(find_col(survey, "truth_y", "truth_y"), "survey truth y", survey)
    px = require_col(find_col(master, "pseudo_x", "pseudo_x"), "pseudo truth x (pt_x)", master)
    py = require_col(find_col(master, "pseudo_y", "pseudo_y"), "pseudo truth y (pt_y)", master)
    bx = find_col(master, "baseline_x", "baseline_x")
    by = find_col(master, "baseline_y", "baseline_y")
    poix = find_col(master, "poi_x", "poi_x")
    poiy = find_col(master, "poi_y", "poi_y")
    loc = find_col(master, "locality_id", "locality_id")

    master[mid] = master[mid].astype(str)
    survey[sid] = survey[sid].astype(str)
    addr = master.groupby(mid, as_index=False).first().copy()

    numeric_cols = [px, py, bx, by, poix, poiy, "pt_k", "pt_spread_m", "clean_visits", "total_visits", "avg_integrity_risk"]
    numeric_cols += STRUCTURED_NUM_CANDIDATES + EXTRA_NUM_CANDIDATES
    for c in [x for x in numeric_cols if x and x in addr.columns]:
        addr[c] = pd.to_numeric(addr[c], errors="coerce")
    survey[tx] = pd.to_numeric(survey[tx], errors="coerce")
    survey[ty] = pd.to_numeric(survey[ty], errors="coerce")

    addr, visit_mode = build_visit_consensus(addr, mid, px, py)
    pseudo_train = addr.loc[addr[[px, py]].notna().all(axis=1)].copy()
    if len(pseudo_train) < MIN_MODEL_ROWS:
        raise RuntimeError(f"Only {len(pseudo_train)} pseudo-truth rows found; need at least {MIN_MODEL_ROWS}.")

    components, locality_pred, global_center, feature_meta = build_component_predictions(
        addr, pseudo_train, px, py, bx, by, poix, poiy, loc
    )
    addr = add_candidate_columns(addr, components)

    dis, cnt = [], []
    for i in range(len(addr)):
        d, c = component_disagreement(components, i)
        dis.append(d); cnt.append(c)
    addr["component_disagreement_m"] = dis
    addr["component_source_count"] = cnt

    locality_lookup = {}
    if loc and loc in addr.columns:
        for i, value in enumerate(addr[loc]):
            if pd.notna(value) and np.all(np.isfinite(locality_pred[i])):
                locality_lookup[str(value)] = locality_pred[i]

    print("Building pseudo-truth OOF NV_grid training examples...")
    router_oof = build_pseudo_oof_router_data(addr, pseudo_train, px, py, bx, by, poix, poiy, loc)
    print("Fitting Architecture 5.2 OOD-aware top-1/top-2 router...")
    router52_pack = tune_and_fit_router_a52(router_oof, loc)
    router52_pack["tuning_trials"].head(30).to_csv(outdir / "architecture52_router_tuning_top30.csv", index=False)

    support_all = build_support_features(pseudo_train, addr, feature_meta["grid_num"], loc)
    for c in support_all.columns:
        addr[c] = support_all[c].to_numpy()
    addr = add_router_meta_features(addr, bx, by)

    r52_xy, r52_weights, r52_risks, r52_adjusted, r52_top1, r52_top2, r52_ood, r52_reason = apply_router_pack_a52(router52_pack, addr)
    addr["router52_pred_x"] = r52_xy[:, 0]
    addr["router52_pred_y"] = r52_xy[:, 1]
    addr["router52_top1"] = r52_top1
    addr["router52_top2"] = r52_top2
    addr["router52_ood"] = r52_ood.astype(int)
    addr["router52_ood_reason"] = r52_reason
    for s in ROUTER_SOURCES:
        addr[f"router52_weight_{s}"] = r52_weights[s]
        addr[f"router52_predicted_risk_{s}_m"] = r52_risks[s]
        addr[f"router52_adjusted_risk_{s}_m"] = r52_adjusted[s]

    survey_keyed = survey.rename(columns={sid: mid}).copy()
    eval_df = survey_keyed.merge(addr, on=mid, how="left", suffixes=("_survey", ""))
    if eval_df["clean_visit_count"].isna().any():
        miss = eval_df.loc[eval_df["clean_visit_count"].isna(), mid].head(10).tolist()
        raise RuntimeError(f"Survey IDs missing from master: {miss}")

    def evidence_tier(r):
        k = int(r["clean_visit_count"])
        if k >= 3: return "V3plus"
        if k == 2: return "V2"
        if k == 1: return "V1"
        has_poi = bool(poix and poiy and pd.notna(r.get(poix)) and pd.notna(r.get(poiy)))
        has_grid = any(pd.notna(r.get(c)) for c in feature_meta["grid_num"])
        if has_grid and has_poi: return "NV_grid_poi"
        if has_grid: return "NV_grid"
        if has_poi: return "NV_poi"
        if loc and pd.notna(r.get(loc)): return "NV_locality"
        return "NV_fallback"

    eval_df["evidence_tier"] = eval_df.apply(evidence_tier, axis=1)
    addr["evidence_tier"] = addr.apply(evidence_tier, axis=1)
    source_names = ["grid", "resid", "extra", "poi", "baseline"]

    def fit_predict_a5_meta(cal_frame: pd.DataFrame, target_frame: pd.DataFrame):
        no_cal = cal_frame.loc[cal_frame["clean_visit_count"] == 0].copy()
        weights = inverse_error_weights(no_cal, source_names, tx, ty)
        base_cal = weighted_component_ensemble(no_cal, weights, source_names)
        gate = fit_tail_gate(no_cal, base_cal, tx, ty)

        no_target = target_frame.loc[target_frame["clean_visit_count"] == 0].copy()
        base_t = weighted_component_ensemble(no_target, weights, source_names)
        gated_t, alpha_t = apply_tail_gate(no_target, base_t, gate)
        gated_t, methods_t = fallback_no_visit(no_target, gated_t, locality_lookup, global_center, loc, bx, by)

        pred = np.full((len(target_frame), 2), np.nan)
        method = np.array([""] * len(target_frame), dtype=object)
        alpha = np.zeros(len(target_frame), float)
        idx_to_pos = {idx: p for p, idx in enumerate(target_frame.index)}

        vis = target_frame["clean_visit_count"] > 0
        pred[vis.to_numpy(), 0] = pd.to_numeric(target_frame.loc[vis, "visit_pin_x"], errors="coerce")
        pred[vis.to_numpy(), 1] = pd.to_numeric(target_frame.loc[vis, "visit_pin_y"], errors="coerce")
        method[vis.to_numpy()] = "visited_weighted_geometric_median"

        for j, idx in enumerate(no_target.index):
            p = idx_to_pos[idx]
            pred[p] = gated_t[j]
            alpha[p] = alpha_t[j]
            method[p] = methods_t[j]
        return pred, method, alpha, weights, gate

    def fit_predict_a52_meta(cal_frame: pd.DataFrame, target_frame: pd.DataFrame):
        pred, method, alpha, weights, gate = fit_predict_a5_meta(cal_frame, target_frame)
        is_nv_grid = (
            (target_frame["clean_visit_count"].to_numpy() == 0) &
            (target_frame["evidence_tier"].astype(str).to_numpy() == "NV_grid")
        )
        rx = pd.to_numeric(target_frame["router52_pred_x"], errors="coerce").to_numpy(float)
        ry = pd.to_numeric(target_frame["router52_pred_y"], errors="coerce").to_numpy(float)
        ok = is_nv_grid & np.isfinite(rx) & np.isfinite(ry)
        pred[ok, 0] = rx[ok]; pred[ok, 1] = ry[ok]
        method[ok] = "NV_grid_A52_OOD_top1_top2_router"
        alpha[ok] = 0.0
        return pred, method, alpha, weights, gate

    strata = np.where(eval_df["clean_visit_count"].to_numpy() > 0, "visited", "no_visit")
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    a5_x = np.full(len(eval_df), np.nan); a5_y = np.full(len(eval_df), np.nan)
    a52_x = np.full(len(eval_df), np.nan); a52_y = np.full(len(eval_df), np.nan)
    a52_r = np.full(len(eval_df), np.nan)
    a52_method = np.array([""] * len(eval_df), dtype=object)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(np.arange(len(eval_df)), strata), 1):
        cal = eval_df.iloc[tr_idx].copy()
        val = eval_df.iloc[va_idx].copy()

        p5, _, _, _, _ = fit_predict_a5_meta(cal, val)
        p52, m52, al52, _, _ = fit_predict_a52_meta(cal, val)
        a5_x[va_idx] = p5[:, 0]; a5_y[va_idx] = p5[:, 1]
        a52_x[va_idx] = p52[:, 0]; a52_y[va_idx] = p52[:, 1]
        a52_method[va_idx] = m52

        
        pcal, _, alcal, _, _ = fit_predict_a52_meta(cal, cal)
        cal["A52_x"] = pcal[:, 0]; cal["A52_y"] = pcal[:, 1]
        cal["tail_gate_alpha"] = alcal
        cal["A52_error_m"] = euclid(cal["A52_x"], cal["A52_y"], cal[tx], cal[ty])
        cal["dist_pred_to_baseline_m"] = euclid(cal["A52_x"], cal["A52_y"], cal[bx], cal[by]) if bx and by else np.nan
        cal["dist_pred_to_poi_m"] = euclid(cal["A52_x"], cal["A52_y"], cal[poix], cal[poiy]) if poix and poiy else np.nan

        val["A52_x"] = p52[:, 0]; val["A52_y"] = p52[:, 1]
        val["tail_gate_alpha"] = al52
        val["dist_pred_to_baseline_m"] = euclid(val["A52_x"], val["A52_y"], val[bx], val[by]) if bx and by else np.nan
        val["dist_pred_to_poi_m"] = euclid(val["A52_x"], val["A52_y"], val[poix], val[poiy]) if poix and poiy else np.nan
        conf = fit_confidence_model(cal, "A52_error_m", loc)
        a52_r[va_idx] = predict_radius(conf, val)

    eval_df["A5_oof_x"] = a5_x; eval_df["A5_oof_y"] = a5_y
    eval_df["A5_oof_error_m"] = euclid(a5_x, a5_y, eval_df[tx], eval_df[ty])
    eval_df["A52_oof_x"] = a52_x; eval_df["A52_oof_y"] = a52_y
    eval_df["A52_oof_error_m"] = euclid(a52_x, a52_y, eval_df[tx], eval_df[ty])
    eval_df["A52_oof_radius_m"] = a52_r
    eval_df["A52_oof_method"] = a52_method
    if bx and by:
        eval_df["baseline_error_m"] = euclid(eval_df[bx], eval_df[by], eval_df[tx], eval_df[ty])

    
    final_pred, _, final_alpha, _, _ = fit_predict_a52_meta(eval_df, eval_df)
    eval_df["A52_final_x"] = final_pred[:, 0]; eval_df["A52_final_y"] = final_pred[:, 1]
    eval_df["A52_final_error_m"] = euclid(eval_df["A52_final_x"], eval_df["A52_final_y"], eval_df[tx], eval_df[ty])
    eval_df["tail_gate_alpha"] = final_alpha
    eval_df["dist_pred_to_baseline_m"] = euclid(eval_df["A52_final_x"], eval_df["A52_final_y"], eval_df[bx], eval_df[by]) if bx and by else np.nan
    eval_df["dist_pred_to_poi_m"] = euclid(eval_df["A52_final_x"], eval_df["A52_final_y"], eval_df[poix], eval_df[poiy]) if poix and poiy else np.nan
    final_conf = fit_confidence_model(eval_df, "A52_final_error_m", loc)

    all_pred, all_method, all_alpha, _, _ = fit_predict_a52_meta(eval_df, addr)
    all_frame = addr.copy()
    all_frame["A52_pin_x"] = all_pred[:, 0]; all_frame["A52_pin_y"] = all_pred[:, 1]
    all_frame["A52_method"] = all_method; all_frame["tail_gate_alpha"] = all_alpha
    all_frame["dist_pred_to_baseline_m"] = euclid(all_frame["A52_pin_x"], all_frame["A52_pin_y"], all_frame[bx], all_frame[by]) if bx and by else np.nan
    all_frame["dist_pred_to_poi_m"] = euclid(all_frame["A52_pin_x"], all_frame["A52_pin_y"], all_frame[poix], all_frame[poiy]) if poix and poiy else np.nan
    all_frame["A52_confidence_radius_m"] = predict_radius(final_conf, all_frame)

    scopes = {
        "OOF_ALL_SURVEYED": eval_df,
        "OOF_VISITED": eval_df.loc[eval_df["clean_visit_count"] > 0],
        "OOF_NO_VISIT": eval_df.loc[eval_df["clean_visit_count"] == 0],
        "OOF_NV_GRID": eval_df.loc[eval_df["evidence_tier"] == "NV_grid"],
        "OOF_NV_GRID_POI": eval_df.loc[eval_df["evidence_tier"] == "NV_grid_poi"],
        "OOF_NV_POI": eval_df.loc[eval_df["evidence_tier"] == "NV_poi"],
    }
    rows = []
    for scope, fr in scopes.items():
        for model, err, rad in [
            ("Architecture_5_reference", "A5_oof_error_m", None),
            ("Architecture_5_2", "A52_oof_error_m", "A52_oof_radius_m"),
            ("commercial_baseline", "baseline_error_m", None),
        ]:
            if err not in fr.columns:
                continue
            r = summarize(fr, err, rad)
            r.update({"scope": scope, "model": model})
            r["errors_over_1000m"] = int((pd.to_numeric(fr[err], errors="coerce") > 1000).sum())
            rows.append(r)
    summary = pd.DataFrame(rows)
    summary.to_csv(outdir / "architecture52_oof_summary.csv", index=False)


    diag_cols = [
        mid, tx, ty, "evidence_tier", "A5_oof_error_m", "A52_oof_error_m", "baseline_error_m",
        "A5_oof_x", "A5_oof_y", "A52_oof_x", "A52_oof_y",
        "router_grid_fields_count", "router_locality_train_count",
        "router_grid_extrapolation_count", "router_grid_extrapolation_score",
        "router_nearest_grid_feature_distance", "router_component_disagreement_m",
        "router52_top1", "router52_top2", "router52_ood", "router52_ood_reason",
    ]
    for s in ROUTER_SOURCES:
        diag_cols += [
            f"cand_{s}_x", f"cand_{s}_y",
            f"router52_weight_{s}", f"router52_predicted_risk_{s}_m", f"router52_adjusted_risk_{s}_m",
        ]
    nv_diag = eval_df.loc[eval_df["evidence_tier"] == "NV_grid", [c for c in diag_cols if c in eval_df.columns]].copy()
    for s in ROUTER_SOURCES:
        cx, cy = f"cand_{s}_x", f"cand_{s}_y"
        ecol = f"actual_error_{s}_m"
        nv_diag[ecol] = np.nan
        if cx in nv_diag.columns and cy in nv_diag.columns:
            ok = nv_diag[[cx, cy, tx, ty]].notna().all(axis=1)
            if ok.any():
                nv_diag.loc[ok, ecol] = euclid(nv_diag.loc[ok, cx], nv_diag.loc[ok, cy], nv_diag.loc[ok, tx], nv_diag.loc[ok, ty])
    expert_err_cols = [f"actual_error_{s}_m" for s in ROUTER_SOURCES]
    nv_diag["oracle_best_expert"] = nv_diag[expert_err_cols].idxmin(axis=1).str.replace("actual_error_", "", regex=False).str.replace("_m", "", regex=False)
    nv_diag["oracle_best_error_m"] = nv_diag[expert_err_cols].min(axis=1)
    nv_diag["A52_minus_A5_error_m"] = nv_diag["A52_oof_error_m"] - nv_diag["A5_oof_error_m"]
    nv_diag = nv_diag.sort_values("A52_oof_error_m", ascending=False)
    nv_diag.to_csv(outdir / "architecture52_nv_grid_diagnostics.csv", index=False)

    pred_cols = [
        mid, tx, ty, "clean_visit_count", "evidence_tier",
        "A5_oof_x", "A5_oof_y", "A5_oof_error_m",
        "A52_oof_x", "A52_oof_y", "A52_oof_error_m", "A52_oof_radius_m", "A52_oof_method",
    ]
    if bx and by:
        pred_cols += [bx, by, "baseline_error_m"]
    pred_cols += [
        "router52_top1", "router52_top2", "router52_ood", "router52_ood_reason",
        "router_component_disagreement_m", "router_grid_extrapolation_score",
        "router_nearest_grid_feature_distance", "router_locality_train_count",
    ]
    for s in ROUTER_SOURCES:
        pred_cols += [f"router52_weight_{s}", f"router52_predicted_risk_{s}_m", f"router52_adjusted_risk_{s}_m"]
    eval_df[[c for c in pred_cols if c in eval_df.columns]].to_csv(outdir / "architecture52_oof_survey_predictions.csv", index=False)

    all_cols = [
        mid, "A52_pin_x", "A52_pin_y", "A52_confidence_radius_m", "A52_method",
        "clean_visit_count", "evidence_tier", "router52_top1", "router52_top2",
        "router52_ood", "router52_ood_reason", "router_grid_extrapolation_score",
        "router_nearest_grid_feature_distance", "router_locality_train_count",
    ]
    for s in ROUTER_SOURCES:
        all_cols += [f"router52_weight_{s}", f"router52_predicted_risk_{s}_m", f"router52_adjusted_risk_{s}_m"]
    if loc: all_cols.append(loc)
    if bx and by: all_cols += [bx, by]
    if poix and poiy: all_cols += [poix, poiy]
    all_frame[[c for c in all_cols if c in all_frame.columns]].to_csv(outdir / "architecture52_all_addresses.csv", index=False)

    expert_summary = []
    for s in ROUTER_SOURCES:
        e = pd.to_numeric(router_oof[f"router_error_{s}_m"], errors="coerce").dropna().to_numpy(float)
        expert_summary.append({
            "source": s, "n": len(e),
            "median_error_m": float(np.median(e)) if len(e) else np.nan,
            "p90_error_m": float(np.quantile(e, .9)) if len(e) else np.nan,
            "cvar90_m": float(a52_cvar(e)) if len(e) else np.nan,
            "mean_error_m": float(np.mean(e)) if len(e) else np.nan,
            "catastrophic_rate": float(np.mean(e > 1000.0)) if len(e) else np.nan,
        })
    pd.DataFrame(expert_summary).to_csv(outdir / "architecture52_router_pseudo_oof_experts.csv", index=False)

    def get_metric(scope, model, col):
        x = summary[(summary["scope"] == scope) & (summary["model"] == model)]
        return float(x.iloc[0][col]) if len(x) else np.nan

    nv_med = get_metric("OOF_NO_VISIT", "Architecture_5_2", "median_error_m")
    nv_p90 = get_metric("OOF_NO_VISIT", "Architecture_5_2", "p90_error_m")
    vis_med = get_metric("OOF_VISITED", "Architecture_5_2", "median_error_m")
    grid_med = get_metric("OOF_NV_GRID", "Architecture_5_2", "median_error_m")
    grid_p90 = get_metric("OOF_NV_GRID", "Architecture_5_2", "p90_error_m")
    a5_nv_p90 = get_metric("OOF_NO_VISIT", "Architecture_5_reference", "p90_error_m")

    config = {
        "architecture": "Architecture_5_2_OOD_aware_top1_top2_expert_router",
        "unchanged_visited_branch": "A2 weighted geometric median",
        "unchanged_non_NV_grid_branch": "Architecture 5 A4 ensemble + A3 disagreement gate",
        "changed_NV_grid_branch": "OOD-aware predicted-risk top-1/top-2 selection among grid/residual-GBM/ExtraTrees/commercial baseline",
        "unchanged_confidence": "A3 90th-quantile model + calibration",
        "router_training": "pseudo-truth OOF simulated no-visit; target POI hidden; risk models cross-validated; survey truth excluded",
        "router_features_num": router52_pack["num"],
        "router_features_cat": router52_pack["cat"],
        "router_priors": router52_pack["priors"],
        "router_params": router52_pack["params"],
        "router_tuning_best": router52_pack["tuning_best"],
        "visit_mode": visit_mode,
        "success_targets": {"no_visit_median_lte_m": 235.0, "no_visit_p90_lt_m": 700.0, "visited_median_lte_m": 20.0},
        "observed": {"no_visit_median_m": nv_med, "no_visit_p90_m": nv_p90, "visited_median_m": vis_med, "nv_grid_median_m": grid_med, "nv_grid_p90_m": grid_p90},
    }
    (outdir / "architecture52_model_config.json").write_text(json.dumps(config, indent=2, default=float))

    report = []
    report.append("=" * 104)
    report.append("CreditNirvana PS3 - Architecture 5.2 OOD-Aware Top-1/Top-2 NV_grid Expert Router")
    report.append("=" * 104)
    report.append(f"Master addresses: {len(addr):,} | surveyed: {len(eval_df):,} | pseudo-truth: {len(pseudo_train):,}")
    report.append(f"Pseudo-OOF router training examples with grid evidence: {len(router_oof):,}")
    report.append(f"Visit evidence mode: {visit_mode}")
    report.append("")
    report.append("WHAT CHANGED FROM ARCHITECTURE 5.1")
    report.append("  Visited: unchanged -> A2 weighted geometric median")
    report.append("  Non-NV_grid no-visit: unchanged -> Architecture 5")
    report.append("  Confidence: unchanged -> A3 quantile + calibration")
    report.append("  NV_grid: four-way blend removed; use OOD-aware top-1/top-2 expert selection")
    report.append("")
    report.append("ARCHITECTURE 5.2 ROUTER")
    report.append(f"  Learned priors: {router52_pack['priors']}")
    report.append(f"  Tuned params: {router52_pack['params']}")
    report.append(f"  Pseudo-OOF tuning best: {router52_pack['tuning_best']}")
    report.append("")
    report.append("PRIMARY OOF COMPARISON")
    report.append(summary.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))
    report.append("")
    report.append("TARGET CHECK")
    report.append(f"  No-visit median <= 235m: {'PASS' if np.isfinite(nv_med) and nv_med <= 235 else 'MISS'} ({nv_med:.2f}m)")
    report.append(f"  No-visit p90 < 700m:    {'PASS' if np.isfinite(nv_p90) and nv_p90 < 700 else 'MISS'} ({nv_p90:.2f}m)")
    report.append(f"  Visited median <= 20m:  {'PASS' if np.isfinite(vis_med) and vis_med <= 20 else 'MISS'} ({vis_med:.2f}m)")
    report.append(f"  A5 no-visit p90 reference: {a5_nv_p90:.2f}m")
    report.append(f"  A5.2 NV_grid median/p90:   {grid_med:.2f}m / {grid_p90:.2f}m")
    report.append("")
    report.append("OUTPUTS")
    report.append("  architecture52_oof_summary.csv")
    report.append("  architecture52_oof_survey_predictions.csv")
    report.append("  architecture52_nv_grid_diagnostics.csv")
    report.append("  architecture52_router_tuning_top30.csv")
    report.append("  architecture52_router_pseudo_oof_experts.csv")
    report.append("  architecture52_all_addresses.csv")
    report.append("  architecture52_model_config.json")
    (outdir / "architecture52_run_report.txt").write_text("\n".join(report))

    print("\n".join(report[:19]))
    print("\nPRIMARY OOF COMPARISON")
    print(summary.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))
    print("\nTARGET CHECK")
    print(f"No-visit median <=235m: {nv_med:.2f}m")
    print(f"No-visit p90 <700m:     {nv_p90:.2f}m")
    print(f"Visited median <=20m:   {vis_med:.2f}m")
    print(f"NV_grid median / p90:   {grid_med:.2f}m / {grid_p90:.2f}m")
    print(f"\nSaved outputs to: {outdir.resolve()}")



if __name__ == "__main__":
    main()


In [ ]:
run_script('cn_architecture52.py')
a52 = pd.read_csv('stage2_arch52/architecture52_all_addresses.csv')
assert len(a52) == len(master) and a52.address_id.is_unique
assert np.isfinite(a52[['A52_pin_x','A52_pin_y']].to_numpy(float)).all()
print('A5.2 predicted pins:',len(a52))

In [ ]:
%%writefile cn_confidence20.py
from __future__ import annotations

import json
import math
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)


# --------------------------------- LOCAL PATHS -----------------------------------------
MASTER_PATH = "ps3_out/ps3_master_processed.csv"
A52_OOF_PATH = "stage2_arch52/architecture52_oof_survey_predictions.csv"
A52_ALL_PATH = "stage2_arch52/architecture52_all_addresses.csv"
OUTPUT_DIR = "stage2_confidence20"

# -------------------------------- EXPERIMENT SETTINGS ----------------------------------
RANDOM_STATE = 42
OUTER_FOLDS = 5
INNER_FOLDS = 4

# Search over raw conditional quantile shapes. Conformal calibration fixes final coverage.
RAW_QUANTILES = [0.70, 0.80, 0.90]
TARGET_COVERAGES = [0.88, 0.90]
MIN_FINE_GROUP_SIZES = [8, 12]

# Coverage guardrails used when choosing the smallest useful radius configuration.
MIN_OVERALL_COVERAGE = 0.88
MIN_VISITED_COVERAGE = 0.88
MIN_NO_VISIT_COVERAGE = 0.88
MIN_NV_GRID_COVERAGE = 0.85
MAX_PREFERRED_OVERALL_COVERAGE = 0.93

# Prevent numerically silly zero/negative radii before calibration.
RAW_RADIUS_FLOOR_M = 10.0
FINAL_RADIUS_FLOOR_M = 10.0

ROUTER_SOURCES = ["grid", "resid", "extra", "baseline"]



def euclid(x1, y1, x2, y2):
    return np.sqrt((np.asarray(x1, float) - np.asarray(x2, float)) ** 2 +
                   (np.asarray(y1, float) - np.asarray(y2, float)) ** 2)


def safe_num(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")


def first_existing(df: pd.DataFrame, names: List[str]) -> Optional[str]:
    for c in names:
        if c in df.columns:
            return c
    return None


def load_csv(path: str) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(
            f"Required file not found: {path}\n"
            f"Run Architecture 5.2 first and make sure this notebook is launched from the project root."
        )
    return pd.read_csv(p)




def _simple_imputer(strategy: str):
    try:
        return SimpleImputer(strategy=strategy, keep_empty_features=True)
    except TypeError:
        return SimpleImputer(strategy=strategy)

def onehot_dense():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=2)
    except TypeError:  # older sklearn
        try:
            return OneHotEncoder(handle_unknown="ignore", sparse=False, min_frequency=2)
        except TypeError:
            return OneHotEncoder(handle_unknown="ignore", sparse=False)


def finite_conformal_quantile(scores: np.ndarray, coverage: float) -> float:
    """Finite-sample split-conformal quantile: ceil((n+1)*coverage)-th order statistic."""
    x = np.asarray(scores, float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return 1.0
    x = np.sort(x)
    k = int(math.ceil((len(x) + 1) * float(coverage)))
    k = min(max(k, 1), len(x))
    return float(x[k - 1])


def coverage_metrics(errors: np.ndarray, radii: np.ndarray) -> Dict[str, float]:
    e = np.asarray(errors, float)
    r = np.asarray(radii, float)
    ok = np.isfinite(e) & np.isfinite(r)
    e, r = e[ok], r[ok]
    if len(e) == 0:
        return {
            "n": 0, "coverage_pct": np.nan, "median_radius_m": np.nan,
            "p90_radius_m": np.nan, "median_error_m": np.nan, "p90_error_m": np.nan,
            "mean_error_m": np.nan, "sharpness_ratio": np.nan,
        }
    return {
        "n": int(len(e)),
        "coverage_pct": float(100 * np.mean(e <= r)),
        "median_radius_m": float(np.median(r)),
        "p90_radius_m": float(np.quantile(r, 0.90)),
        "median_error_m": float(np.median(e)),
        "p90_error_m": float(np.quantile(e, 0.90)),
        "mean_error_m": float(np.mean(e)),
        "sharpness_ratio": float(np.median(r) / max(np.median(e), 1e-9)),
    }


def normalize_boolish(s: pd.Series) -> pd.Series:
    if s.dtype == bool:
        return s.astype(int)
    st = s.astype(str).str.strip().str.lower()
    mp = {"true": 1, "1": 1, "yes": 1, "y": 1, "false": 0, "0": 0, "no": 0, "n": 0, "nan": 0, "none": 0}
    return st.map(mp).fillna(0).astype(int)


def selected_value(frame: pd.DataFrame, selector_col: str, prefix: str, suffix: str = "") -> pd.Series:
    out = pd.Series(np.nan, index=frame.index, dtype=float)
    if selector_col not in frame.columns:
        return out
    sel = frame[selector_col].astype(str)
    for src in ROUTER_SOURCES:
        col = f"{prefix}{src}{suffix}"
        if col in frame.columns:
            m = sel.eq(src)
            out.loc[m] = safe_num(frame.loc[m, col])
    return out




def merge_master_features(pred: pd.DataFrame, master: pd.DataFrame, address_id: str) -> pd.DataFrame:
    master_keep = [
        address_id,
        "pt_spread_m", "total_visits", "clean_visits", "flagged_visits",
        "avg_integrity_risk", "agents_involved", "locality_conf", "n_fields_found",
        "script_kn_frac", "script_dev_frac", "is_village_style",
        "locality_method", "town_method", "base_precision", "lm_primary",
        "rel_no_type", "rel_context", "poi_x", "poi_y", "geocoder_x", "geocoder_y",
        "house_no", "cross_no", "main_no", "gali_no", "road_no", "ward_no",
    ]
    master_keep = [c for c in master_keep if c in master.columns]
    # Avoid duplicate columns already emitted by A5.2.
    extras = [address_id] + [c for c in master_keep if c != address_id and c not in pred.columns]
    return pred.merge(master[extras].drop_duplicates(address_id), on=address_id, how="left")


def build_confidence_frame(df: pd.DataFrame, production: bool = False) -> pd.DataFrame:
    f = df.copy()


    if production:
        pin_x = first_existing(f, ["A52_pin_x", "pin_x"])
        pin_y = first_existing(f, ["A52_pin_y", "pin_y"])
        old_r = first_existing(f, ["A52_confidence_radius_m", "confidence_radius_m"])
        method = first_existing(f, ["A52_method", "method"])
    else:
        pin_x = first_existing(f, ["A52_oof_x", "A52_pin_x", "pin_x"])
        pin_y = first_existing(f, ["A52_oof_y", "A52_pin_y", "pin_y"])
        old_r = first_existing(f, ["A52_oof_radius_m", "A52_confidence_radius_m", "confidence_radius_m"])
        method = first_existing(f, ["A52_oof_method", "A52_method", "method"])

    if pin_x is None or pin_y is None:
        raise KeyError("Could not find Architecture 5.2 pin coordinate columns.")

    f["pin_x"] = safe_num(f[pin_x])
    f["pin_y"] = safe_num(f[pin_y])
    f["current_radius_m"] = safe_num(f[old_r]) if old_r else np.nan
    f["pin_method"] = f[method].astype(str) if method else "unknown"

    
    err_col = first_existing(f, ["A52_oof_error_m", "error_m", "actual_error_m"])
    if err_col:
        f["actual_error_m"] = safe_num(f[err_col])

    
    cvc = first_existing(f, ["clean_visit_count", "clean_visits", "pt_k"])
    if cvc is None:
        f["clean_visit_count"] = 0
    else:
        f["clean_visit_count"] = safe_num(f[cvc]).fillna(0)

    f["is_visited"] = (f["clean_visit_count"] > 0).astype(int)
    f["broad_conf_group"] = np.where(f["is_visited"].eq(1), "Visited", "NoVisit")

    evidence = f["evidence_tier"].astype(str) if "evidence_tier" in f.columns else pd.Series("NV_other", index=f.index)
    conf_tier = []
    for idx, row in f.iterrows():
        k = float(row.get("clean_visit_count", 0) or 0)
        if k > 0:
            if k <= 1:
                conf_tier.append("V1")
            elif k <= 2:
                conf_tier.append("V2")
            else:
                conf_tier.append("V3plus")
        else:
            et = str(evidence.loc[idx])
            if et in {"NV_grid", "NV_grid_poi", "NV_poi", "NV_locality", "NV_fallback"}:
                conf_tier.append(et)
            else:
                conf_tier.append("NV_other")
    f["confidence_tier"] = conf_tier

    
    if "router52_ood" in f.columns:
        f["router52_ood_num"] = normalize_boolish(f["router52_ood"])
    else:
        f["router52_ood_num"] = 0

   
    f["top1_predicted_risk_m"] = selected_value(f, "router52_top1", "router52_predicted_risk_", "_m")
    f["top2_predicted_risk_m"] = selected_value(f, "router52_top2", "router52_predicted_risk_", "_m")
    f["top1_adjusted_risk_m"] = selected_value(f, "router52_top1", "router52_adjusted_risk_", "_m")
    f["top2_adjusted_risk_m"] = selected_value(f, "router52_top2", "router52_adjusted_risk_", "_m")

    risk_cols = [f"router52_adjusted_risk_{s}_m" for s in ROUTER_SOURCES if f"router52_adjusted_risk_{s}_m" in f.columns]
    pred_risk_cols = [f"router52_predicted_risk_{s}_m" for s in ROUTER_SOURCES if f"router52_predicted_risk_{s}_m" in f.columns]
    weight_cols = [f"router52_weight_{s}" for s in ROUTER_SOURCES if f"router52_weight_{s}" in f.columns]

    if risk_cols:
        rr = f[risk_cols].apply(pd.to_numeric, errors="coerce")
        f["router_min_adjusted_risk_m"] = rr.min(axis=1)
        f["router_second_adjusted_risk_m"] = rr.apply(lambda r: np.sort(r.dropna().to_numpy(float))[1] if r.notna().sum() >= 2 else np.nan, axis=1)
        f["router_median_adjusted_risk_m"] = rr.median(axis=1)
        f["router_max_adjusted_risk_m"] = rr.max(axis=1)
        f["router_adjusted_risk_spread_m"] = rr.max(axis=1) - rr.min(axis=1)
    if pred_risk_cols:
        rp = f[pred_risk_cols].apply(pd.to_numeric, errors="coerce")
        f["router_min_predicted_risk_m"] = rp.min(axis=1)
        f["router_predicted_risk_spread_m"] = rp.max(axis=1) - rp.min(axis=1)

    f["router_risk_gap_m"] = f["top2_adjusted_risk_m"] - f["top1_adjusted_risk_m"]
    f["router_risk_ratio"] = f["top2_adjusted_risk_m"] / f["top1_adjusted_risk_m"].clip(lower=1.0)

    if weight_cols:
        w = f[weight_cols].apply(pd.to_numeric, errors="coerce").fillna(0).clip(lower=0)
        row_sum = w.sum(axis=1).replace(0, np.nan)
        wn = w.div(row_sum, axis=0)
        f["router_weight_max"] = wn.max(axis=1)
        f["router_weight_entropy"] = -(wn.where(wn > 0) * np.log(wn.where(wn > 0))).sum(axis=1)
    else:
        f["router_weight_max"] = np.nan
        f["router_weight_entropy"] = np.nan

    # Geometric distances available at prediction time.
    if {"geocoder_x", "geocoder_y"}.issubset(f.columns):
        f["pin_to_baseline_m"] = euclid(f["pin_x"], f["pin_y"], safe_num(f["geocoder_x"]), safe_num(f["geocoder_y"]))
    if {"poi_x", "poi_y"}.issubset(f.columns):
        ok = f[["poi_x", "poi_y"]].notna().all(axis=1)
        f["pin_to_poi_m"] = np.nan
        f.loc[ok, "pin_to_poi_m"] = euclid(
            f.loc[ok, "pin_x"], f.loc[ok, "pin_y"], safe_num(f.loc[ok, "poi_x"]), safe_num(f.loc[ok, "poi_y"])
        )
        if {"geocoder_x", "geocoder_y"}.issubset(f.columns):
            f["baseline_to_poi_m"] = np.nan
            f.loc[ok, "baseline_to_poi_m"] = euclid(
                safe_num(f.loc[ok, "geocoder_x"]), safe_num(f.loc[ok, "geocoder_y"]),
                safe_num(f.loc[ok, "poi_x"]), safe_num(f.loc[ok, "poi_y"])
            )

    return f




NUMERIC_CANDIDATES = [
    "clean_visit_count", "pt_spread_m", "total_visits", "clean_visits", "flagged_visits",
    "avg_integrity_risk", "agents_involved", "locality_conf", "n_fields_found",
    "script_kn_frac", "script_dev_frac", "is_village_style", "current_radius_m",
    "router52_ood_num", "router_grid_extrapolation_score",
    "router_nearest_grid_feature_distance", "router_locality_train_count",
    "top1_predicted_risk_m", "top2_predicted_risk_m", "top1_adjusted_risk_m", "top2_adjusted_risk_m",
    "router_min_adjusted_risk_m", "router_second_adjusted_risk_m", "router_median_adjusted_risk_m",
    "router_max_adjusted_risk_m", "router_adjusted_risk_spread_m",
    "router_min_predicted_risk_m", "router_predicted_risk_spread_m",
    "router_risk_gap_m", "router_risk_ratio", "router_weight_max", "router_weight_entropy",
    "pin_to_baseline_m", "pin_to_poi_m", "baseline_to_poi_m",
    "house_no", "cross_no", "main_no", "gali_no", "road_no", "ward_no",
]

CATEGORICAL_CANDIDATES = [
    "confidence_tier", "evidence_tier", "pin_method", "router52_top1", "router52_top2",
    "router52_ood", "router52_ood_reason", "locality_method", "town_method", "base_precision",
    "lm_primary", "rel_no_type", "rel_context",
]


def usable_features(train: pd.DataFrame, production: pd.DataFrame) -> Tuple[List[str], List[str]]:
    common = set(train.columns) & set(production.columns)
    nums, cats = [], []
    for c in NUMERIC_CANDIDATES:
        if c in common and safe_num(train[c]).notna().sum() >= 3:
            nums.append(c)
    for c in CATEGORICAL_CANDIDATES:
        if c in common and train[c].notna().sum() >= 3 and train[c].astype(str).nunique(dropna=True) >= 2:
            cats.append(c)
    return nums, cats


def coerce_selected_numeric_features(
    train: pd.DataFrame,
    production: pd.DataFrame,
    num_cols: List[str],
) -> Tuple[pd.DataFrame, pd.DataFrame, List[Dict[str, object]]]:
    """
    Force every selected numeric feature to a true numeric dtype before it reaches
    SimpleImputer/StandardScaler.

    Address-derived fields can legitimately contain tokens such as "206/2".
    Those values must NOT be passed to a median imputer as strings. We coerce
    non-numeric tokens to NaN rather than inventing a numeric interpretation.
    """
    train = train.copy()
    production = production.copy()
    diagnostics = []

    for c in num_cols:
        for label, df in [("OOF", train), ("production", production)]:
            raw = df[c].copy()
            converted = pd.to_numeric(raw, errors="coerce")

            raw_nonmissing = raw.notna()
            newly_missing = raw_nonmissing & converted.isna()

            if newly_missing.any():
                examples = (
                    raw.loc[newly_missing]
                    .astype(str)
                    .drop_duplicates()
                    .head(8)
                    .tolist()
                )
                diagnostics.append({
                    "dataset": label,
                    "feature": c,
                    "n_non_numeric_coerced_to_nan": int(newly_missing.sum()),
                    "examples": examples,
                })

            df[c] = converted.astype(float)

    return train, production, diagnostics


def make_pipeline(num_cols: List[str], cat_cols: List[str], alpha: float, seed: int) -> Pipeline:
    transformers = []
    if num_cols:
        num_pipe = Pipeline([
            ("imputer", _simple_imputer("median")),
            ("scale", StandardScaler()),
        ])
        transformers.append(("num", num_pipe, num_cols))
    if cat_cols:
        cat_pipe = Pipeline([
            ("imputer", _simple_imputer("most_frequent")),
            ("onehot", onehot_dense()),
        ])
        transformers.append(("cat", cat_pipe, cat_cols))

    prep = ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=0.0)
    reg = HistGradientBoostingRegressor(
        loss="quantile",
        quantile=float(alpha),
        max_iter=120,
        learning_rate=0.05,
        max_leaf_nodes=15,
        max_depth=3,
        min_samples_leaf=5,
        l2_regularization=1.0,
        early_stopping=False,
        random_state=seed,
    )
    return Pipeline([("prep", prep), ("model", reg)])


def active_features(sub: pd.DataFrame, num_cols: List[str], cat_cols: List[str]):
    nums = [c for c in num_cols if c in sub.columns and safe_num(sub[c]).notna().sum() >= 3]
    cats = [c for c in cat_cols if c in sub.columns and sub[c].notna().sum() >= 3 and sub[c].astype(str).nunique(dropna=True) >= 2]
    return nums, cats


def fit_broad_models(train_df: pd.DataFrame, alpha: float, num_cols: List[str], cat_cols: List[str], seed: int):
    models = {}
    global_nums, global_cats = active_features(train_df, num_cols, cat_cols)
    global_model = make_pipeline(global_nums, global_cats, alpha, seed)
    global_model.fit(train_df[global_nums + global_cats], train_df["actual_error_m"])

    for g in ["Visited", "NoVisit"]:
        sub = train_df[train_df["broad_conf_group"] == g]
        if len(sub) < 15:
            models[g] = (global_model, global_nums, global_cats)
            continue
        nums, cats = active_features(sub, num_cols, cat_cols)
        model = make_pipeline(nums, cats, alpha, seed + (17 if g == "Visited" else 29))
        model.fit(sub[nums + cats], sub["actual_error_m"])
        models[g] = (model, nums, cats)
    return models


def predict_broad_models(models, df: pd.DataFrame) -> pd.Series:
    out = pd.Series(np.nan, index=df.index, dtype=float)
    for g in ["Visited", "NoVisit"]:
        idx = df.index[df["broad_conf_group"] == g]
        if len(idx) == 0:
            continue
        model, nums, cats = models[g]
        pred = model.predict(df.loc[idx, nums + cats])
        out.loc[idx] = np.maximum(pred, RAW_RADIUS_FLOOR_M)
    return out


def inner_oof_raw(train_outer: pd.DataFrame, alpha: float, num_cols: List[str], cat_cols: List[str], seed: int) -> pd.Series:
    ystrat = train_outer["broad_conf_group"].astype(str)
    min_count = int(ystrat.value_counts().min())
    n_splits = max(2, min(INNER_FOLDS, min_count))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    raw = pd.Series(np.nan, index=train_outer.index, dtype=float)
    arr = np.arange(len(train_outer))
    for j, (tr_pos, va_pos) in enumerate(skf.split(arr, ystrat.to_numpy())):
        tr_idx = train_outer.index[tr_pos]
        va_idx = train_outer.index[va_pos]
        models = fit_broad_models(train_outer.loc[tr_idx], alpha, num_cols, cat_cols, seed + 100 + j)
        raw.loc[va_idx] = predict_broad_models(models, train_outer.loc[va_idx])
    return raw



def make_calibration_map(cal_df: pd.DataFrame, raw_radius: pd.Series, target_coverage: float, min_fine_n: int):
    work = cal_df.copy()
    work["raw_radius"] = raw_radius.reindex(work.index).clip(lower=RAW_RADIUS_FLOOR_M)
    work["ratio"] = work["actual_error_m"] / work["raw_radius"].clip(lower=RAW_RADIUS_FLOOR_M)

    factors = {"fine": {}, "broad": {}, "global": 1.0}
    factor_rows = []

    global_factor = finite_conformal_quantile(work["ratio"].to_numpy(float), target_coverage)
    factors["global"] = global_factor
    factor_rows.append({"level": "global", "group": "ALL", "n": len(work), "factor": global_factor})

    for g, sub in work.groupby("broad_conf_group"):
        if len(sub) >= min_fine_n:
            q = finite_conformal_quantile(sub["ratio"].to_numpy(float), target_coverage)
        else:
            q = global_factor
        factors["broad"][str(g)] = q
        factor_rows.append({"level": "broad", "group": str(g), "n": len(sub), "factor": q})

    for g, sub in work.groupby("confidence_tier"):
        broad = str(sub["broad_conf_group"].iloc[0])
        if len(sub) >= min_fine_n:
            q = finite_conformal_quantile(sub["ratio"].to_numpy(float), target_coverage)
            level = "fine"
        else:
            q = factors["broad"].get(broad, global_factor)
            level = "broad_fallback"
        factors["fine"][str(g)] = q
        factor_rows.append({"level": level, "group": str(g), "n": len(sub), "factor": q})

    return factors, pd.DataFrame(factor_rows)


def apply_calibration(df: pd.DataFrame, raw_radius: pd.Series, factors):
    final = pd.Series(np.nan, index=df.index, dtype=float)
    factor_used = pd.Series(np.nan, index=df.index, dtype=float)
    level_used = pd.Series("global", index=df.index, dtype=object)

    for idx, row in df.iterrows():
        fine = str(row["confidence_tier"])
        broad = str(row["broad_conf_group"])
        if fine in factors["fine"]:
            q = factors["fine"][fine]
            lvl = "fine_or_fallback"
        elif broad in factors["broad"]:
            q = factors["broad"][broad]
            lvl = "broad"
        else:
            q = factors["global"]
            lvl = "global"
        factor_used.loc[idx] = q
        level_used.loc[idx] = lvl
        final.loc[idx] = max(FINAL_RADIUS_FLOOR_M, float(raw_radius.loc[idx]) * float(q))
    return final, factor_used, level_used



def strat_labels(df: pd.DataFrame) -> pd.Series:
    # Use fine tiers when frequent enough for 5 folds, otherwise broad tier.
    counts = df["confidence_tier"].value_counts()
    return df.apply(
        lambda r: r["confidence_tier"] if counts.get(r["confidence_tier"], 0) >= OUTER_FOLDS else r["broad_conf_group"],
        axis=1,
    ).astype(str)


def build_outer_cache(df: pd.DataFrame, alpha: float, num_cols: List[str], cat_cols: List[str]):
    labels = strat_labels(df)
    skf = StratifiedKFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    cache = []
    arr = np.arange(len(df))
    for fold, (tr_pos, te_pos) in enumerate(skf.split(arr, labels.to_numpy())):
        tr_idx = df.index[tr_pos]
        te_idx = df.index[te_pos]
        tr = df.loc[tr_idx]
        te = df.loc[te_idx]

        raw_train_oof = inner_oof_raw(tr, alpha, num_cols, cat_cols, RANDOM_STATE + 1000 * (fold + 1))
        final_models = fit_broad_models(tr, alpha, num_cols, cat_cols, RANDOM_STATE + 5000 + fold)
        raw_test = predict_broad_models(final_models, te)

        cache.append({
            "fold": fold,
            "train_idx": tr_idx,
            "test_idx": te_idx,
            "raw_train_oof": raw_train_oof,
            "raw_test": raw_test,
        })
    return cache


def evaluate_config(df: pd.DataFrame, cache, target_cov: float, min_fine_n: int):
    final = pd.Series(np.nan, index=df.index, dtype=float)
    raw_all = pd.Series(np.nan, index=df.index, dtype=float)
    factor_all = pd.Series(np.nan, index=df.index, dtype=float)
    level_all = pd.Series("", index=df.index, dtype=object)
    fold_all = pd.Series(-1, index=df.index, dtype=int)
    factor_tables = []

    for c in cache:
        tr = df.loc[c["train_idx"]]
        te = df.loc[c["test_idx"]]
        factors, ft = make_calibration_map(tr, c["raw_train_oof"], target_cov, min_fine_n)
        fr, fu, lu = apply_calibration(te, c["raw_test"], factors)
        final.loc[te.index] = fr
        raw_all.loc[te.index] = c["raw_test"]
        factor_all.loc[te.index] = fu
        level_all.loc[te.index] = lu
        fold_all.loc[te.index] = c["fold"]
        ft["outer_fold"] = c["fold"]
        factor_tables.append(ft)

    return final, raw_all, factor_all, level_all, fold_all, pd.concat(factor_tables, ignore_index=True)


def scope_mask(df: pd.DataFrame, scope: str):
    if scope == "ALL":
        return pd.Series(True, index=df.index)
    if scope == "VISITED":
        return df["is_visited"].eq(1)
    if scope == "NO_VISIT":
        return df["is_visited"].eq(0)
    if scope == "NV_GRID":
        return df["confidence_tier"].eq("NV_grid")
    return df["confidence_tier"].eq(scope)


def summarize_radius_model(df: pd.DataFrame, radii: pd.Series, model_name: str):
    rows = []
    scopes = ["ALL", "VISITED", "NO_VISIT", "NV_GRID", "V1", "V2", "V3plus", "NV_grid_poi", "NV_poi", "NV_locality", "NV_fallback"]
    for scope in scopes:
        m = scope_mask(df, scope)
        if not m.any():
            continue
        met = coverage_metrics(df.loc[m, "actual_error_m"], radii.loc[m])
        met.update({"scope": scope, "model": model_name})
        rows.append(met)
    return pd.DataFrame(rows)


def config_selection_row(summary: pd.DataFrame, alpha, target_cov, min_n):
    def get(scope, col):
        x = summary[summary["scope"] == scope]
        return float(x.iloc[0][col]) if len(x) else np.nan

    oc = get("ALL", "coverage_pct") / 100.0
    vc = get("VISITED", "coverage_pct") / 100.0
    nc = get("NO_VISIT", "coverage_pct") / 100.0
    gc = get("NV_GRID", "coverage_pct") / 100.0 if (summary["scope"] == "NV_GRID").any() else np.nan

    mr = get("ALL", "median_radius_m")
    nvr = get("NO_VISIT", "median_radius_m")
    gr = get("NV_GRID", "median_radius_m") if (summary["scope"] == "NV_GRID").any() else nvr

    feasible = (
        oc >= MIN_OVERALL_COVERAGE and
        vc >= MIN_VISITED_COVERAGE and
        nc >= MIN_NO_VISIT_COVERAGE and
        (not np.isfinite(gc) or gc >= MIN_NV_GRID_COVERAGE)
    )
    preferred = feasible and oc <= MAX_PREFERRED_OVERALL_COVERAGE

    # Radius efficiency objective; coverage shortfalls receive huge penalty.
    radius_score = mr + 0.25 * nvr + 0.15 * gr
    shortfall = (
        max(0, MIN_OVERALL_COVERAGE - oc) +
        max(0, MIN_VISITED_COVERAGE - vc) +
        max(0, MIN_NO_VISIT_COVERAGE - nc) +
        (max(0, MIN_NV_GRID_COVERAGE - gc) if np.isfinite(gc) else 0)
    )
    overcoverage_penalty = 1200 * max(0, oc - 0.92)
    score = radius_score + 20000 * shortfall + overcoverage_penalty

    return {
        "raw_quantile": alpha,
        "target_coverage": target_cov,
        "min_fine_group_n": min_n,
        "overall_coverage_pct": 100 * oc,
        "visited_coverage_pct": 100 * vc,
        "no_visit_coverage_pct": 100 * nc,
        "nv_grid_coverage_pct": 100 * gc if np.isfinite(gc) else np.nan,
        "overall_median_radius_m": mr,
        "visited_median_radius_m": get("VISITED", "median_radius_m"),
        "no_visit_median_radius_m": nvr,
        "nv_grid_median_radius_m": gr,
        "feasible": bool(feasible),
        "preferred": bool(preferred),
        "selection_score": float(score),
    }



def main():
    outdir = Path(OUTPUT_DIR)
    outdir.mkdir(parents=True, exist_ok=True)

    master = load_csv(MASTER_PATH)
    oof = load_csv(A52_OOF_PATH)
    all_addr = load_csv(A52_ALL_PATH)

    address_id = first_existing(oof, ["address_id", "addr_id", "id"])
    if address_id is None:
        raise KeyError(f"Could not identify address_id in {A52_OOF_PATH}. Columns: {list(oof.columns)}")
    if address_id not in master.columns or address_id not in all_addr.columns:
        raise KeyError(f"Address id '{address_id}' must exist in master and Architecture 5.2 production outputs.")

    oof = merge_master_features(oof, master, address_id)
    all_addr = merge_master_features(all_addr, master, address_id)

    train = build_confidence_frame(oof, production=False)
    prod = build_confidence_frame(all_addr, production=True)

    if "actual_error_m" not in train.columns or train["actual_error_m"].notna().sum() < 50:
        raise KeyError("Architecture 5.2 OOF file must contain A52_oof_error_m (or equivalent actual error).")
    train = train[train["actual_error_m"].notna()].copy()

    num_cols, cat_cols = usable_features(train, prod)
    if not num_cols and not cat_cols:
        raise RuntimeError("No usable confidence features were found in both OOF and production files.")


    train, prod, numeric_coercion_diagnostics = coerce_selected_numeric_features(
        train, prod, num_cols
    )

    print("=" * 100)
    print("CreditNirvana PS3 — Confidence 2.0")
    print("=" * 100)
    print(f"OOF surveyed rows: {len(train)} | Production addresses: {len(prod)}")
    print(f"Numeric confidence features ({len(num_cols)}): {num_cols}")
    print(f"Categorical confidence features ({len(cat_cols)}): {cat_cols}")

    if numeric_coercion_diagnostics:
        print("\nMixed numeric tokens safely coerced to NaN before sklearn:")
        for d in numeric_coercion_diagnostics:
            print(
                f"  - {d['dataset']} / {d['feature']}: "
                f"{d['n_non_numeric_coerced_to_nan']} value(s); "
                f"examples={d['examples']}"
            )
        print(
            "These are treated as missing numeric values rather than being "
            "incorrectly interpreted as numbers."
        )

    print("Pins are frozen: Architecture 5.2 coordinates are never retrained or modified here.\n")

   
    current_summary = summarize_radius_model(train, train["current_radius_m"], "Current_A52_confidence")

    # Search configs. Cache expensive model fits per raw quantile.
    candidate_rows = []
    result_store = {}
    cache_by_alpha = {}

    for alpha in RAW_QUANTILES:
        print(f"Building nested OOF quantile cache for raw quantile={alpha:.2f} ...")
        cache = build_outer_cache(train, alpha, num_cols, cat_cols)
        cache_by_alpha[alpha] = cache
        for target_cov in TARGET_COVERAGES:
            for min_n in MIN_FINE_GROUP_SIZES:
                radii, raw, factor, level, fold, factor_table = evaluate_config(train, cache, target_cov, min_n)
                summary = summarize_radius_model(train, radii, "Confidence_2_0_candidate")
                row = config_selection_row(summary, alpha, target_cov, min_n)
                candidate_rows.append(row)
                result_store[(alpha, target_cov, min_n)] = {
                    "radii": radii, "raw": raw, "factor": factor, "level": level,
                    "fold": fold, "factor_table": factor_table, "summary": summary,
                }

    search = pd.DataFrame(candidate_rows).sort_values(["preferred", "feasible", "selection_score"], ascending=[False, False, True])
    search.to_csv(outdir / "confidence20_candidate_search.csv", index=False)

    # Prefer configs within the desired coverage window, then any feasible config, then best score.
    preferred = search[search["preferred"]]
    feasible = search[search["feasible"]]
    if len(preferred):
        best_row = preferred.sort_values("selection_score").iloc[0]
    elif len(feasible):
        best_row = feasible.sort_values("selection_score").iloc[0]
    else:
        best_row = search.sort_values("selection_score").iloc[0]

    best_key = (float(best_row["raw_quantile"]), float(best_row["target_coverage"]), int(best_row["min_fine_group_n"]))
    best = result_store[best_key]

    # Final OOF comparison.
    best_summary = summarize_radius_model(train, best["radii"], "Confidence_2_0")
    combined_summary = pd.concat([current_summary, best_summary], ignore_index=True)
    combined_summary.to_csv(outdir / "confidence20_oof_summary.csv", index=False)

    oof_out_cols = [address_id, "actual_error_m", "confidence_tier", "broad_conf_group", "evidence_tier", "clean_visit_count", "pin_x", "pin_y", "current_radius_m"]
    for c in ["router52_top1", "router52_top2", "router52_ood", "router52_ood_reason", "top1_adjusted_risk_m", "top2_adjusted_risk_m", "router_risk_ratio", "router_locality_train_count", "router_nearest_grid_feature_distance", "router_grid_extrapolation_score"]:
        if c in train.columns:
            oof_out_cols.append(c)
    oof_out = train[[c for c in oof_out_cols if c in train.columns]].copy()
    oof_out["conf20_raw_radius_m"] = best["raw"]
    oof_out["conf20_calibration_factor"] = best["factor"]
    oof_out["conf20_calibration_level"] = best["level"]
    oof_out["conf20_radius_m"] = best["radii"]
    oof_out["conf20_covered"] = (oof_out["actual_error_m"] <= oof_out["conf20_radius_m"]).astype(int)
    oof_out["current_covered"] = (oof_out["actual_error_m"] <= oof_out["current_radius_m"]).astype(int)
    oof_out["radius_reduction_m"] = oof_out["current_radius_m"] - oof_out["conf20_radius_m"]
    oof_out.to_csv(outdir / "confidence20_oof_predictions.csv", index=False)
    best["factor_table"].to_csv(outdir / "confidence20_oof_calibration_factors.csv", index=False)

    # Train production Confidence 2.0 on all surveyed OOF errors.
    best_alpha, best_target, best_min_n = best_key

    # Cross-fitted raw predictions over all training rows -> calibration ratios for production.
    labels = strat_labels(train)
    skf = StratifiedKFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=RANDOM_STATE + 777)
    cv_raw = pd.Series(np.nan, index=train.index, dtype=float)
    arr = np.arange(len(train))
    for fold, (tr_pos, va_pos) in enumerate(skf.split(arr, labels.to_numpy())):
        tr_idx = train.index[tr_pos]
        va_idx = train.index[va_pos]
        models = fit_broad_models(train.loc[tr_idx], best_alpha, num_cols, cat_cols, RANDOM_STATE + 8000 + fold)
        cv_raw.loc[va_idx] = predict_broad_models(models, train.loc[va_idx])

    prod_factors, prod_factor_table = make_calibration_map(train, cv_raw, best_target, best_min_n)
    prod_factor_table.to_csv(outdir / "confidence20_production_calibration_factors.csv", index=False)

    final_models = fit_broad_models(train, best_alpha, num_cols, cat_cols, RANDOM_STATE + 9999)
    prod_raw = predict_broad_models(final_models, prod)
    prod_radius, prod_factor, prod_level = apply_calibration(prod, prod_raw, prod_factors)

    prod_out_cols = [address_id, "pin_x", "pin_y", "confidence_tier", "broad_conf_group", "evidence_tier", "clean_visit_count", "pin_method", "current_radius_m"]
    for c in ["router52_top1", "router52_top2", "router52_ood", "router52_ood_reason", "top1_adjusted_risk_m", "top2_adjusted_risk_m", "router_risk_ratio", "router_locality_train_count", "router_nearest_grid_feature_distance", "router_grid_extrapolation_score"]:
        if c in prod.columns:
            prod_out_cols.append(c)
    prod_out = prod[[c for c in prod_out_cols if c in prod.columns]].copy()
    prod_out["confidence20_raw_radius_m"] = prod_raw
    prod_out["confidence20_calibration_factor"] = prod_factor
    prod_out["confidence20_calibration_level"] = prod_level
    prod_out["confidence20_radius_m"] = prod_radius
    prod_out.to_csv(outdir / "confidence20_all_addresses.csv", index=False)

    # Group-level OOF diagnostics.
    group_rows = []
    for group_col in ["confidence_tier", "broad_conf_group", "evidence_tier"]:
        if group_col not in train.columns:
            continue
        for g, idx in train.groupby(group_col).groups.items():
            idx = list(idx)
            oldm = coverage_metrics(train.loc[idx, "actual_error_m"], train.loc[idx, "current_radius_m"])
            newm = coverage_metrics(train.loc[idx, "actual_error_m"], best["radii"].loc[idx])
            group_rows.append({
                "group_type": group_col, "group": str(g), "n": len(idx),
                "current_coverage_pct": oldm["coverage_pct"], "conf20_coverage_pct": newm["coverage_pct"],
                "current_median_radius_m": oldm["median_radius_m"], "conf20_median_radius_m": newm["median_radius_m"],
                "median_error_m": newm["median_error_m"],
                "radius_reduction_pct": 100 * (oldm["median_radius_m"] - newm["median_radius_m"]) / max(oldm["median_radius_m"], 1e-9),
            })
    group_df = pd.DataFrame(group_rows)
    group_df.to_csv(outdir / "confidence20_group_diagnostics.csv", index=False)

    # Report.
    def row(scope, model):
        x = combined_summary[(combined_summary["scope"] == scope) & (combined_summary["model"] == model)]
        return x.iloc[0].to_dict() if len(x) else {}

    report_lines = []
    report_lines.append("=" * 100)
    report_lines.append("CreditNirvana PS3 - Confidence 2.0")
    report_lines.append("Architecture 5.2 pins are frozen; only the radius estimator was changed.")
    report_lines.append("=" * 100)
    report_lines.append(f"OOF surveyed rows: {len(train)} | Production addresses: {len(prod)}")
    report_lines.append(f"Selected raw quantile: {best_alpha}")
    report_lines.append(f"Selected conformal target coverage: {best_target}")
    report_lines.append(f"Selected minimum fine-tier calibration size: {best_min_n}")
    report_lines.append("")
    report_lines.append("OOF COMPARISON")
    for scope in ["ALL", "VISITED", "NO_VISIT", "NV_GRID"]:
        old = row(scope, "Current_A52_confidence")
        new = row(scope, "Confidence_2_0")
        if not new:
            continue
        report_lines.append(
            f"{scope:10s} | Current coverage={old.get('coverage_pct', np.nan):6.2f}% radius={old.get('median_radius_m', np.nan):8.2f}m "
            f"| Conf2 coverage={new.get('coverage_pct', np.nan):6.2f}% radius={new.get('median_radius_m', np.nan):8.2f}m "
            f"| actual median error={new.get('median_error_m', np.nan):8.2f}m"
        )
    report_lines.append("")
    report_lines.append("SUCCESS TARGETS")
    report_lines.append("  Coverage target: roughly 88-92% overall / visited / no-visit; NV_grid guardrail >=85% because n=27.")
    report_lines.append("  Radius targets: overall <600m, visited <200m, no-visit <800m, NV_grid <900m when coverage remains honest.")
    report_lines.append("")
    report_lines.append("OUTPUTS")
    for fn in [
        "confidence20_oof_summary.csv", "confidence20_oof_predictions.csv", "confidence20_group_diagnostics.csv",
        "confidence20_candidate_search.csv", "confidence20_oof_calibration_factors.csv",
        "confidence20_production_calibration_factors.csv", "confidence20_all_addresses.csv", "confidence20_model_config.json",
    ]:
        report_lines.append(f"  {fn}")

    (outdir / "confidence20_run_report.txt").write_text("\n".join(report_lines), encoding="utf-8")

    config = {
        "architecture": "Confidence_2_0_evidence_conditioned_quantile_plus_hierarchical_conformal",
        "pin_model": "Architecture_5_2_frozen",
        "selected_raw_quantile": best_alpha,
        "selected_target_coverage": best_target,
        "selected_min_fine_group_n": best_min_n,
        "numeric_features": num_cols,
        "categorical_features": cat_cols,
        "coverage_guardrails": {
            "overall_min": MIN_OVERALL_COVERAGE,
            "visited_min": MIN_VISITED_COVERAGE,
            "no_visit_min": MIN_NO_VISIT_COVERAGE,
            "nv_grid_min": MIN_NV_GRID_COVERAGE,
            "preferred_overall_max": MAX_PREFERRED_OVERALL_COVERAGE,
        },
        "production_calibration_factors": prod_factors,
    }
    (outdir / "confidence20_model_config.json").write_text(json.dumps(config, indent=2, default=float), encoding="utf-8")

    print("\nBEST CONFIDENCE 2.0 CONFIG")
    print(best_row.to_string())
    print("\nOOF SUMMARY")
    show = combined_summary[combined_summary["scope"].isin(["ALL", "VISITED", "NO_VISIT", "NV_GRID"])][
        ["scope", "model", "n", "coverage_pct", "median_radius_m", "p90_radius_m", "median_error_m", "p90_error_m", "sharpness_ratio"]
    ]
    print(show.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))
    print(f"\nSaved outputs to: {outdir.resolve()}")



if __name__ == "__main__":
    main()


In [ ]:
run_script('cn_confidence20.py')
conf = pd.read_csv('stage2_confidence20/confidence20_all_addresses.csv')
assert len(conf) == len(master) and conf.address_id.is_unique
assert (pd.to_numeric(conf.confidence20_radius_m,errors='coerce')>0).all()
print('Confidence 2.0 calibrated radii:',len(conf))

## 4. Landmark directions and conservative publication policy



In [ ]:
%%writefile cn_directions_and_integrations.py
"""Directions and offline integration examples, built only from raw PS3 data and fitted predictions."""
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT=Path('.')
RAW=Path('dataset')
OUT=Path('submission_outputs')

def nfloat(value):
    try: return float(value)
    except (TypeError, ValueError): return float('nan')

def label(value):
    if pd.isna(value) or str(value).strip()=='':return ''
    return str(value).replace('_',' ').strip()

def street_parts(r, localities):
    pieces=[]
    for key,noun in [('house_no','No.'),('cross_no','Cross'),('main_no','Main'),('gali_no','Gali'),('road_no','Road'),('ward_no','Ward')]:
        v=r.get(key)
        if pd.isna(v) or str(v).strip()=='': continue
        try:
            k=float(v);v=str(int(k)) if k.is_integer() else str(v)
        except (ValueError,TypeError):v=str(v)
        pieces.append(f'{noun} {v}')
    loc=r.get('locality_id')
    if pd.notna(loc) and str(loc) in localities:
        pieces.append(localities[str(loc)])
    else:
        town=r.get('town_id')
        if pd.notna(town):pieces.append(f'Town {town}')
    return ', '.join(pieces[:5])

def build_outputs(raw_dir, root=Path('.')):
    raw_dir=Path(raw_dir);root=Path(root);out=root/'submission_outputs';out.mkdir(parents=True,exist_ok=True)
    master=pd.read_csv(root/'ps3_out'/'ps3_master_processed.csv')
    pin=pd.read_csv(root/'stage2_arch52'/'architecture52_all_addresses.csv')
    conf=pd.read_csv(root/'stage2_confidence20'/'confidence20_all_addresses.csv')
    poi=pd.read_csv(raw_dir/'landmarks_poi.csv')
    loc=pd.read_csv(raw_dir/'localities.csv')
    survey=pd.read_csv(raw_dir/'surveyed_addresses.csv')
    visits=pd.read_csv(root/'ps3_out'/'visit_features.csv')
    accounts=pd.read_csv(raw_dir/'accounts.csv')
    if not all(v.address_id.is_unique for v in [master,pin,conf]):raise ValueError('Duplicate address ID')
    cols=['address_id','A52_pin_x','A52_pin_y','A52_method','evidence_tier']
    data=master.merge(pin[cols],on='address_id',validate='one_to_one',how='left')
    data=data.merge(conf[['address_id','confidence20_radius_m','confidence_tier','confidence20_calibration_level']],on='address_id',validate='one_to_one',how='left')
    if len(data)!=len(master):raise AssertionError('A master address was lost')
    data=data.rename(columns={'A52_pin_x':'pin_x','A52_pin_y':'pin_y','confidence20_radius_m':'confidence_radius_m'})
    if (not np.isfinite(data[['pin_x','pin_y','confidence_radius_m']].to_numpy(float)).all()
        or (data.confidence_radius_m <= 0).any()):
        raise AssertionError('Missing pins or invalid confidence radii')
    poimap=poi.set_index('poi_id').to_dict(orient='index')
    locmap=loc.set_index('locality_id').locality_name.astype(str).to_dict()
    types=[];sources=[];quals=[];texts=[];active_poi=[];flags=[];anchor_names=[];distances=[]
    for r in data.to_dict('records'):
        street=street_parts(r,locmap);tail=f' — {street}' if street else ''
        poid=r.get('poi_id'); family=label(r.get('lm_primary'))
        confidence=nfloat(r['confidence_radius_m'])
        d=np.hypot(nfloat(r['pin_x'])-nfloat(r.get('poi_x')), nfloat(r['pin_y'])-nfloat(r.get('poi_y')))
        valid_named=pd.notna(poid) and str(poid) in poimap
        suspect=bool(valid_named and nfloat(r.get('clean_visits'))>0 and d >= max(1000,2*confidence))
        if valid_named and not suspect:
            record=poimap[str(poid)]
            name=str(record.get('name','')).strip() or label(record.get('landmark_type'))
            ptype=label(record.get('landmark_type'))
            t=f'Reference POI: {name} ({ptype}){tail}. Confirm location using the pin; landmark identity is provisional.'
            direction='named_poi_provisional';quality='identity_unverified';source='stage1_matched_poi';assigned=poid
        elif family:
            phrase=('Review needed: ' if suspect else '')+f'Address mentions {family}{tail}. Specific landmark identity is unverified.'
            t=phrase;direction='family_only';quality='family_reference';source='parser_family';assigned=np.nan
        else:
            t=(f'Use the corrected pin{tail}.' if street else 'Use the corrected pin; address has limited reference details.')
            direction='street_only';quality='address_only';source='structured_address';assigned=np.nan
            if suspect:t='Review needed: '+t
        types.append(direction);sources.append(source);quals.append(quality);texts.append(t)
        active_poi.append(assigned);flags.append(suspect);anchor_names.append(str(poimap.get(str(poid),{}).get('name','')) if valid_named else '')
        distances.append(d if valid_named else np.nan)
    data['direction_type']=types;data['direction_quality']=quals;data['landmark_direction']=texts
    data['anchor_source']=sources;data['anchor_poi_id']=active_poi
    data['manual_review']=flags;data['raw_matched_poi_name']=anchor_names
    data['raw_matched_poi_distance_m']=distances
    useful=['address_id','account_id','town_id','pin_x','pin_y','confidence_radius_m','A52_method',
            'evidence_tier','confidence_tier','landmark_direction','direction_type','direction_quality',
            'anchor_source','anchor_poi_id','manual_review','clean_visits','flagged_visits','avg_integrity_risk']
    final=data[useful].copy();final.to_csv(out/'final_predictions.csv',index=False)
    review=data.loc[data.manual_review,['address_id','account_id','poi_id','lm_primary','raw_matched_poi_name',
                                        'raw_matched_poi_distance_m','confidence_radius_m','clean_visits']]
    review.to_csv(out/'named_poi_review_queue.csv',index=False)
    # Local, offline planning features; routing is straight-line only without a road graph.
    planner=final[['address_id','account_id','town_id','pin_x','pin_y','confidence_radius_m','direction_type','clean_visits','flagged_visits']].copy()
    planner['location_uncertainty_score']=np.clip(np.log1p(planner.confidence_radius_m)/np.log1p(5000),0,1)
    planner['verification_priority']=np.clip(.75*planner.location_uncertainty_score + .25*(planner.clean_visits.eq(0)),0,1)
    planner['route_pin_reliable']=planner.confidence_radius_m.le(250) & (~planner.direction_type.eq('street_only'))
    planner.to_csv(out/'visit_planner_features.csv',index=False)
    # Geocoding inputs to an RPC/PS2 model. These are features, not PS2 predictions.
    ps2=final[['address_id','account_id','confidence_radius_m','clean_visits','flagged_visits','direction_type','avg_integrity_risk']].copy()
    ps2=ps2.merge(accounts[['account_id','town_id','portfolio','bucket_start','dpd_start']],on='account_id',how='left',validate='many_to_one')
    ps2['location_confidence_proxy']=1/(1+ps2.confidence_radius_m/500)
    ps2.to_csv(out/'ps2_location_features.csv',index=False)
    oof=pd.read_csv(root/'stage2_arch52'/'architecture52_oof_survey_predictions.csv')
    c_oof=pd.read_csv(root/'stage2_confidence20'/'confidence20_oof_predictions.csv')
    def summarize(s):
        a=pd.to_numeric(s,errors='coerce').dropna();return {'median_m':float(a.median()),'p90_m':float(a.quantile(.9)),'mean_m':float(a.mean())}
    pin_error=summarize(oof['A52_oof_error_m'])
    baseline_error=summarize(oof['baseline_error_m'])
    err_col=next((c for c in ['actual_error_m','A52_oof_error_m'] if c in c_oof.columns),None)
    rad_col=next((c for c in ['conf20_radius_m','confidence20_radius_m','confidence_radius_m'] if c in c_oof.columns),None)
    radius_cov=float((pd.to_numeric(c_oof[err_col],errors='coerce')<=pd.to_numeric(c_oof[rad_col],errors='coerce')).mean()) if err_col and rad_col else None
    metrics={'address_count':int(len(final)),'survey_count':int(len(survey)),
             'pin_oof':pin_error,'baseline_oof':baseline_error,'confidence_oof_coverage':radius_cov,
             'confidence_production_median_radius_m':float(final.confidence_radius_m.median()),
             'direction_counts':final.direction_type.value_counts().to_dict(),
             'direction_note':'Named POIs are provisional, family-only directions do not claim POI identity.',
             'flagged_named_poi_cases':int(final.manual_review.sum()),
             'planner_note':'Visit Planner export is feature interface and straight-line heuristic; no routing uplift estimated.',
             'ps2_note':'PS2 export is a feature interface; no Right-Party Contact outcome model trained.',
             'self_learning_note':'The notebook re-fits from raw visits when rerun; see retrospective feedback demonstration.'}
    (out/'summary_metrics.json').write_text(json.dumps(metrics,indent=2,ensure_ascii=False))
    return final,planner,ps2,metrics

if __name__=='__main__':
    print('Import build_outputs(raw_dir) from the notebook.')


In [ ]:
from cn_directions_and_integrations import build_outputs
final, planner, ps2_features, metrics = build_outputs(raw_dir, ROOT)
print('Final predictions:',final.shape)
print('Direction tiers:',metrics['direction_counts'])
print('Named-POI reviews:',metrics['flagged_named_poi_cases'])
display(final[['address_id','pin_x','pin_y','confidence_radius_m','landmark_direction','direction_type']].head(8))

In [ ]:
assert len(final)==len(master)==len(a52)==len(conf)
assert final.address_id.is_unique
assert np.isfinite(final[['pin_x','pin_y','confidence_radius_m']].to_numpy(float)).all()
assert final.confidence_radius_m.gt(0).all()
joined=final.merge(a52[['address_id','A52_pin_x','A52_pin_y']],on='address_id',validate='one_to_one')
assert np.allclose(joined.pin_x,joined.A52_pin_x) and np.allclose(joined.pin_y,joined.A52_pin_y)
joined=final.merge(conf[['address_id','confidence20_radius_m']],on='address_id',validate='one_to_one')
assert np.allclose(joined.confidence_radius_m,joined.confidence20_radius_m)
print('All row-count, finite-value and frozen-output consistency checks passed.')
print('Commercial baseline OOF error:', metrics['baseline_oof'])
print('A5.2 OOF pin error:', metrics['pin_oof'])
print('Confidence 2.0 OOF coverage:',metrics['confidence_oof_coverage'])
print('Actual direction types:',metrics['direction_counts'])

## 6. Field visit feedback — retrospective update example

The following illustrates the *mechanism* for absorbing one additional **clean** visit. We hold out a historically recorded clean visit for demonstration, estimate a consensus with earlier observations, then add the withheld observation. This is **not a prospective field test**, and it does not claim that the trained A5.2 router updated itself online. The full batch system re-learns when new raw visits are appended and the notebook is rerun.

In [ ]:
visits = pd.read_csv('ps3_out/visit_features.csv')
valid = visits.loc[(visits.tier=='clean') & np.isfinite(visits.checkin_x) & np.isfinite(visits.checkin_y)]
counts = valid.groupby('address_id').size().sort_values(ascending=False)
example_id = counts[counts>=3].index[0]
points = valid.loc[valid.address_id.eq(example_id),['checkin_x','checkin_y']].to_numpy(float)

def geometric_median(coords, n_iter=100):
    guess=np.median(coords,axis=0)
    for _ in range(n_iter):
        d=np.linalg.norm(coords-guess,axis=1)
        if np.any(d<1e-9): return coords[np.argmin(d)]
        new=np.sum(coords/np.maximum(d[:,None],1e-6),axis=0)/np.sum(1/np.maximum(d,1e-6))
        if np.linalg.norm(new-guess)<1e-7:break
        guess=new
    return guess

before=geometric_median(points[:-1]);after=geometric_median(points)
print('Historical address:',example_id,'| clean visits before / after:',len(points)-1,'/',len(points))
print('Earlier consensus:',np.round(before,2),'Updated consensus:',np.round(after,2))
print('Retrospective shift (m):',round(float(np.linalg.norm(after-before)),2))

## 7. Integration examples — Visit Planner and PS2

These CSV exports can be consumed by the existing systems. The Visit Planner receives location uncertainty, coordinates, and verification priority. The PS2 Right-Party Contact model receives **location-quality input features** only. We do not claim real road-route gains or RPC uplift without those separate outcomes/models.

In [ ]:
print('Visit Planner interface:')
display(planner[['address_id','pin_x','pin_y','confidence_radius_m','verification_priority','route_pin_reliable']].head())
print('PS2 location feature interface:')
display(ps2_features[['address_id','account_id','confidence_radius_m','location_confidence_proxy','avg_integrity_risk']].head())
# Nearest-neighbour route illustration with Euclidean distance (no street network or speed estimates).
subset=planner.loc[planner.town_id.eq(planner.town_id.value_counts().index[0])].head(8).copy()
remaining=subset.to_dict('records')
order=[remaining.pop(0)]
while remaining:
    last=order[-1]
    j=int(np.argmin([np.hypot(row['pin_x']-last['pin_x'],row['pin_y']-last['pin_y']) for row in remaining]))
    order.append(remaining.pop(j))
print('Illustrative straight-line visit order:', ' → '.join(r['address_id'] for r in order))

## 8. Manager quality dashboards

These charts are generated offline from the model's actual output: accuracy versus the baseline, confidence calibration, evidence tiers, and integrity flags. They serve as dashboard-ready exhibits, not as a separate web application.

In [ ]:
from pathlib import Path
out=Path('submission_outputs')
summary=metrics
fig, ax=plt.subplots(figsize=(7,4))
vals=[summary['baseline_oof']['median_m'],summary['pin_oof']['median_m'],
      summary['baseline_oof']['p90_m'],summary['pin_oof']['p90_m']]
ax.bar(['Baseline median','A5.2 median','Baseline p90','A5.2 p90'], vals)
ax.set_ylabel('Surveyed location error (m)');ax.set_title('Pin quality — out-of-fold evaluation')
fig.tight_layout();fig.savefig(out/'manager_pin_quality.png',dpi=150);plt.show()

fig,axes=plt.subplots(1,2,figsize=(11,4))
final.direction_type.value_counts().plot.bar(ax=axes[0],title='Direction evidence tiers')
axes[0].set_ylabel('Address count');axes[0].tick_params(axis='x',rotation=25)
final.confidence_radius_m.clip(upper=3000).plot.hist(ax=axes[1],bins=28,title='Confidence radius distribution (capped at 3 km)')
axes[1].set_xlabel('Radius (m)')
fig.tight_layout();fig.savefig(out/'manager_directions_and_radii.png',dpi=150);plt.show()
print('Charts saved in',out)

## 9. Deliverables, boundaries, and reproducibility

**Generated outputs** under `submission_outputs/`:

- `final_predictions.csv` — corrected pin, calibrated radius, directions and evidence labels for every address.
- `named_poi_review_queue.csv` — questionable POI relations deliberately suppressed from confident direction text.
- `visit_planner_features.csv` — coordinates, uncertainty, verification priority and straight-line route readiness.
- `ps2_location_features.csv` — joinable location-quality features for PS2; not PS2 predictions.
- `summary_metrics.json`, `manager_pin_quality.png`, `manager_directions_and_radii.png`.

